# Imports

In [1]:
import numpy as np
import matplotlib
from matplotlib import rc
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import scipy
from scipy.ndimage import shift
from scipy.ndimage import gaussian_filter
import scipy.io as sio
from scipy.io import readsav
import csv

#below are ones that I added
import pandas as pd
from bokeh.io import push_notebook, show, output_notebook
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, Label, LabelSet,BoxAnnotation, Range1d
import sys
from scipy.interpolate import LSQUnivariateSpline, UnivariateSpline
output_notebook()

#Global Variables (This is bad, do not emulate my bad programming)
NORMALIZATION_INDEX = 800 #420 #I dunno pick one

Loading BokehJS ...

## Data Load-In

In [2]:
N1_R1_A = readsav('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/Mendoza_Venus_data_19sep2025/venus_1118_a_ortho_19sep25.sav')
N1_R1_B = readsav('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/Mendoza_Venus_data_19sep2025/venus_1118_b_ortho_19sep25.sav')
N1_R1_C = readsav('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/Mendoza_Venus_data_19sep2025/venus_1118_c_ortho_19sep25.sav')
N1_R1_tot = readsav('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/Mendoza_Venus_data_19sep2025/venus_1118_total_ortho_19sep25.sav')

N2_R1_A = readsav('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/Mendoza_Venus_data_21sep2025/venus_1118_a_ortho_21sep25.sav')
N2_R1_B = readsav('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/Mendoza_Venus_data_21sep2025/venus_1118_b_ortho_21sep25.sav')
N2_R1_C = readsav('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/Mendoza_Venus_data_21sep2025/venus_1118_c_ortho_21sep25.sav')
N2_R1_tot = readsav('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/Mendoza_Venus_data_21sep2025/venus_1118_total_ortho_21sep25.sav')

In [3]:
N1_R2_A = readsav('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/Mendoza_Venus_data_19sep2025/venus_1207_a_ortho_19sep25.sav')
N1_R2_B = readsav('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/Mendoza_Venus_data_19sep2025/venus_1207_b_ortho_19sep25.sav')
N1_R2_tot = readsav('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/Mendoza_Venus_data_19sep2025/venus_1207_total_ortho_19sep25.sav')

N2_R2_A = readsav('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/Mendoza_Venus_data_21sep2025/venus_1207_a_ortho_21sep25.sav')
N2_R2_B = readsav('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/Mendoza_Venus_data_21sep2025/venus_1207_b_ortho_21sep25.sav')
N2_R2_C = readsav('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/Mendoza_Venus_data_21sep2025/venus_1207_c_ortho_21sep25.sav')
N2_R2_tot = readsav('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/Mendoza_Venus_data_21sep2025/venus_1207_total_ortho_21sep25.sav')

In [4]:
#Figure out what we got
print(N1_R1_tot.keys())
print(N2_R1_tot.mapgrid)

dict_keys(['spec_cube', 'sky', 'wavenumber', 'hdr', 'latitude', 'longitude', 'airmass', 'vel_map', 'mapgrid', 'inttime', 'map_pixel_scale', 'contmap', 'emismap'])
[-20.5 -20.  -19.5 -19.  -18.5 -18.  -17.5 -17.  -16.5 -16.  -15.5 -15.
 -14.5 -14.  -13.5 -13.  -12.5 -12.  -11.5 -11.  -10.5 -10.   -9.5  -9.
  -8.5  -8.   -7.5  -7.   -6.5  -6.   -5.5  -5.   -4.5  -4.   -3.5  -3.
  -2.5  -2.   -1.5  -1.   -0.5   0.    0.5   1.    1.5   2.    2.5   3.
   3.5   4.    4.5   5.    5.5   6.    6.5   7.    7.5   8.    8.5   9.
   9.5  10.   10.5  11.   11.5  12.   12.5  13.   13.5  14.   14.5  15.
  15.5  16.   16.5  17.   17.5  18.   18.5  19.   19.5  20.   20.5]


In [5]:
print(N1_R2_tot.keys())
print(N2_R2_tot.mapgrid)

dict_keys(['spec_cube', 'sky', 'wavenumber', 'hdr', 'latitude', 'longitude', 'airmass', 'vel_map', 'mapgrid', 'inttime', 'map_pixel_scale', 'contmap', 'emismap'])
[-20.5 -20.  -19.5 -19.  -18.5 -18.  -17.5 -17.  -16.5 -16.  -15.5 -15.
 -14.5 -14.  -13.5 -13.  -12.5 -12.  -11.5 -11.  -10.5 -10.   -9.5  -9.
  -8.5  -8.   -7.5  -7.   -6.5  -6.   -5.5  -5.   -4.5  -4.   -3.5  -3.
  -2.5  -2.   -1.5  -1.   -0.5   0.    0.5   1.    1.5   2.    2.5   3.
   3.5   4.    4.5   5.    5.5   6.    6.5   7.    7.5   8.    8.5   9.
   9.5  10.   10.5  11.   11.5  12.   12.5  13.   13.5  14.   14.5  15.
  15.5  16.   16.5  17.   17.5  18.   18.5  19.   19.5  20.   20.5]


In [6]:
#Print the wavenumber ranges
print("Region 1 Night 1:")
print(str(N1_R1_tot['wavenumber'][0]) + ' - ' + str(N1_R1_tot['wavenumber'][-1]))
print("Region 1 Night 2:")
print(str(N2_R1_tot['wavenumber'][0]) + ' - ' + str(N2_R1_tot['wavenumber'][-1]))

print("Region 2 Night 1:")
print(str(N1_R2_tot['wavenumber'][0]) + ' - ' + str(N1_R2_tot['wavenumber'][-1]))
print("Region 2 Night 2:")
print(str(N2_R2_tot['wavenumber'][0]) + ' - ' + str(N2_R2_tot['wavenumber'][-1]))

Region 1 Night 1:
1113.9566362863977 - 1122.815079266173
Region 1 Night 2:
1114.6205838457513 - 1122.8187838770396
Region 2 Night 1:
1204.0046390186249 - 1210.2899295889044
Region 2 Night 2:
1203.3513422910116 - 1210.2995432767207


### Headers

In [7]:
print(str(N1_R1_tot['hdr']))

[b'filename= ven.7014                                                             '
 b'object  = Venus                                                                '
 b'PI      = Mendoza         ' b'PID     = 2025B098        '
 b'objtype = targ            '
 b'feature = PH3                                                                  '
 b'waveno0 = 1118                                                                 '
 b'order   = 6                                                                    '
 b'temp    =  286.5                                                               '
 b'tau     =                                                                      '
 b'humidity= 22.5                                                                 '
 b'obsmode = scan                                                                 '
 b'telstep = 0.50 E   0.00 N                                                      '
 b'offset  = -11.00 E   -3.00 N                                     

## Line-by-Line Load In

In [8]:
#import line by line (assuming gathered from HITRAN)
df=pd.read_fwf('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/line_by_lines/H2O_all.txt',header=None) 
wn=df.loc[:,1]
abund=df.loc[:,2]
split_df=pd.concat([wn,abund],axis=1)
split_array=split_df.to_numpy()
#print(split_array)

#get desired wavenumbers

mol_lambda1=[]
abund_lambda1=[]
for a in range(0,len(split_array),1):
    if split_array[a,0] > 1114.7114576040406 and split_array[a,0] < 1123.696475590839:
    #if split_array[a,0] > 1121 and split_array[a,0] < 1123:
        if split_array[a,1] > 1e-23: # and split_array[a,1] < 1e-22: #remove the and for the regular limit
            mol_lambda1.append(split_array[a,0]) #get list of wavenumber locations
            abund_lambda1.append(split_array[a,1]) # get corresponding strengths

print(len(mol_lambda1))
range1_array=np.stack((mol_lambda1, abund_lambda1),axis=-1) #make locations and strengths one array
line_locations_region_1 = np.array(mol_lambda1) #make it match previous naming structure

2


In [9]:
#import line by line (assuming gathered from HITRAN)
df=pd.read_fwf('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/line_by_lines/O3_all.txt',header=None) 
wn=df.loc[:,1]
abund=df.loc[:,2]
split_df=pd.concat([wn,abund],axis=1)
split_array=split_df.to_numpy()
#print(split_array)

#get desired wavenumbers

#Region 1:
two_mol_lambda1=[]
two_abund_lambda1=[]
for a in range(0,len(split_array),1):
    if split_array[a,0] > 1114.7114576040406 and split_array[a,0] < 1123.696475590839:
    #if split_array[a,0] > 1117 and split_array[a,0] < 1118 or split_array[a,0] > 1119 and split_array[a,0] < 1119.33:
        if split_array[a,1] > 1e-22: # and split_array[a,1] < 1e-22: #remove the and for the regular limit
            two_mol_lambda1.append(split_array[a,0]) #get list of wavenumber locations
            two_abund_lambda1.append(split_array[a,1]) # get corresponding strengths

print(len(two_mol_lambda1))
two_range1_array=np.stack((two_mol_lambda1, two_abund_lambda1),axis=-1) #make locations and strengths one array
two_line_locations_region_1 = np.array(two_mol_lambda1) #make it match previous naming structure

43


## Cross-Section Load-In

In [10]:
SO2_cross_section_wavenumber=[]
SO2_cross_section=[]

with open('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/cross_sections/SO2_cross_section.csv') as cross_section_file:  
    plots = csv.reader(cross_section_file, delimiter = ',') 
      
    for row in plots: 
        SO2_cross_section_wavenumber.append(float(row[0]))
        SO2_cross_section.append(float(row[1]))
        
SO2_cross_section_normalized = SO2_cross_section / np.median(SO2_cross_section)

# Global Spectra

## Variables

### Region One

In [11]:
disk_N1_R1_A =np.argwhere(N1_R1_A.airmass>0)
disk_N1_R1_B =np.argwhere(N1_R1_B.airmass>0)
disk_N1_R1_C =np.argwhere(N1_R1_C.airmass>0)
disk_N1_R1_tot =np.argwhere(N1_R1_tot.airmass>0)

disk_N2_R1_A =np.argwhere(N2_R1_A.airmass>0)
disk_N2_R1_B =np.argwhere(N2_R1_B.airmass>0)
disk_N2_R1_C =np.argwhere(N2_R1_C.airmass>0)
disk_N2_R1_tot =np.argwhere(N2_R1_tot.airmass>0)

In [12]:
vlambda_N1_R1_A = []
vflux_N1_R1_A = []
for i in range(len(N1_R1_A['wavenumber'])):
    vlambda_N1_R1_A.append(N1_R1_A['wavenumber'][i])
    sum_N1_R1_A=0
    for j in range(len(disk_N1_R1_A)): #disk[j,0]:
        row=disk_N1_R1_A[j,0]
        col=disk_N1_R1_A[j,1]
        sum_N1_R1_A = sum_N1_R1_A + N1_R1_A['spec_cube'][int(row),int(col),int(i)]
    vflux_N1_R1_A.append(sum_N1_R1_A)

slambda_N1_R1_A = []
sflux_N1_R1_A = []

for i in range(len(N1_R1_A['wavenumber'])):
    slambda_N1_R1_A.append(N1_R1_A['wavenumber'][i])
    sum_N1_R1_A=0
    for j in range(len(disk_N1_R1_A)): #disk[j,0]:
        row=disk_N1_R1_A[j,0]
        col=disk_N1_R1_A[j,1]
        sum_N1_R1_A = sum_N1_R1_A + N1_R1_A['sky'][int(row),int(col),int(i)]
    sflux_N1_R1_A.append(sum_N1_R1_A)

In [13]:
vlambda_N1_R1_B = []
vflux_N1_R1_B = []
for i in range(len(N1_R1_B['wavenumber'])):
    vlambda_N1_R1_B.append(N1_R1_B['wavenumber'][i])
    sum_N1_R1_B=0
    for j in range(len(disk_N1_R1_B)): #disk[j,0]:
        row=disk_N1_R1_B[j,0]
        col=disk_N1_R1_B[j,1]
        sum_N1_R1_B = sum_N1_R1_B + N1_R1_B['spec_cube'][int(row),int(col),int(i)]
    vflux_N1_R1_B.append(sum_N1_R1_B)

slambda_N1_R1_B = []
sflux_N1_R1_B = []

for i in range(len(N1_R1_B['wavenumber'])):
    slambda_N1_R1_B.append(N1_R1_B['wavenumber'][i])
    sum_N1_R1_B=0
    for j in range(len(disk_N1_R1_B)): #disk[j,0]:
        row=disk_N1_R1_B[j,0]
        col=disk_N1_R1_B[j,1]
        sum_N1_R1_B = sum_N1_R1_B + N1_R1_B['sky'][int(row),int(col),int(i)]
    sflux_N1_R1_B.append(sum_N1_R1_B)

In [14]:
vlambda_N1_R1_C = []
vflux_N1_R1_C = []
for i in range(len(N1_R1_C['wavenumber'])):
    vlambda_N1_R1_C.append(N1_R1_C['wavenumber'][i])
    sum_N1_R1_C=0
    for j in range(len(disk_N1_R1_C)): #disk[j,0]:
        row=disk_N1_R1_C[j,0]
        col=disk_N1_R1_C[j,1]
        sum_N1_R1_C = sum_N1_R1_C + N1_R1_C['spec_cube'][int(row),int(col),int(i)]
    vflux_N1_R1_C.append(sum_N1_R1_C)

slambda_N1_R1_C = []
sflux_N1_R1_C = []

for i in range(len(N1_R1_C['wavenumber'])):
    slambda_N1_R1_C.append(N1_R1_A['wavenumber'][i])
    sum_N1_R1_C=0
    for j in range(len(disk_N1_R1_C)): #disk[j,0]:
        row=disk_N1_R1_C[j,0]
        col=disk_N1_R1_C[j,1]
        sum_N1_R1_C = sum_N1_R1_C + N1_R1_C['sky'][int(row),int(col),int(i)]
    sflux_N1_R1_C.append(sum_N1_R1_C)

In [15]:
vlambda_N1_R1_tot = []
vflux_N1_R1_tot = []
for i in range(len(N1_R1_tot['wavenumber'])):
    vlambda_N1_R1_tot.append(N1_R1_tot['wavenumber'][i])
    sum_N1_R1_tot=0
    for j in range(len(disk_N1_R1_tot)): #disk[j,0]:
        row=disk_N1_R1_tot[j,0]
        col=disk_N1_R1_tot[j,1]
        sum_N1_R1_tot = sum_N1_R1_tot + N1_R1_tot['spec_cube'][int(row),int(col),int(i)]
    vflux_N1_R1_tot.append(sum_N1_R1_tot)

slambda_N1_R1_tot = []
sflux_N1_R1_tot = []

for i in range(len(N1_R1_tot['wavenumber'])):
    slambda_N1_R1_tot.append(N1_R1_tot['wavenumber'][i])
    sum_N1_R1_tot=0
    for j in range(len(disk_N1_R1_tot)): #disk[j,0]:
        row=disk_N1_R1_tot[j,0]
        col=disk_N1_R1_tot[j,1]
        sum_N1_R1_tot = sum_N1_R1_tot + N1_R1_tot['sky'][int(row),int(col),int(i)]
    sflux_N1_R1_tot.append(sum_N1_R1_tot)

In [16]:
vlambda_N2_R1_A = []
vflux_N2_R1_A = []
for i in range(len(N2_R1_A['wavenumber'])):
    vlambda_N2_R1_A.append(N2_R1_A['wavenumber'][i])
    sum_N2_R1_A=0
    for j in range(len(disk_N2_R1_A)): #disk[j,0]:
        row=disk_N2_R1_A[j,0]
        col=disk_N2_R1_A[j,1]
        sum_N2_R1_A = sum_N2_R1_A + N2_R1_A['spec_cube'][int(row),int(col),int(i)]
    vflux_N2_R1_A.append(sum_N2_R1_A)

slambda_N2_R1_A = []
sflux_N2_R1_A = []

for i in range(len(N2_R1_A['wavenumber'])):
    slambda_N2_R1_A.append(N2_R1_A['wavenumber'][i])
    sum_N2_R1_A=0
    for j in range(len(disk_N2_R1_A)): #disk[j,0]:
        row=disk_N2_R1_A[j,0]
        col=disk_N2_R1_A[j,1]
        sum_N2_R1_A = sum_N2_R1_A + N2_R1_A['sky'][int(row),int(col),int(i)]
    sflux_N2_R1_A.append(sum_N2_R1_A)

In [17]:
vlambda_N2_R1_B = []
vflux_N2_R1_B = []
for i in range(len(N2_R1_B['wavenumber'])):
    vlambda_N2_R1_B.append(N2_R1_B['wavenumber'][i])
    sum_N2_R1_B=0
    for j in range(len(disk_N2_R1_B)): #disk[j,0]:
        row=disk_N2_R1_B[j,0]
        col=disk_N2_R1_B[j,1]
        sum_N2_R1_B = sum_N2_R1_B + N2_R1_B['spec_cube'][int(row),int(col),int(i)]
    vflux_N2_R1_B.append(sum_N2_R1_B)

slambda_N2_R1_B = []
sflux_N2_R1_B = []

for i in range(len(N2_R1_B['wavenumber'])):
    slambda_N2_R1_B.append(N2_R1_B['wavenumber'][i])
    sum_N2_R1_B=0
    for j in range(len(disk_N2_R1_B)): #disk[j,0]:
        row=disk_N2_R1_B[j,0]
        col=disk_N2_R1_B[j,1]
        sum_N2_R1_B = sum_N2_R1_B + N2_R1_B['sky'][int(row),int(col),int(i)]
    sflux_N2_R1_B.append(sum_N2_R1_B)

In [18]:
vlambda_N2_R1_C = []
vflux_N2_R1_C = []
for i in range(len(N2_R1_C['wavenumber'])):
    vlambda_N2_R1_C.append(N2_R1_C['wavenumber'][i])
    sum_N2_R1_C=0
    for j in range(len(disk_N2_R1_C)): #disk[j,0]:
        row=disk_N2_R1_C[j,0]
        col=disk_N2_R1_C[j,1]
        sum_N2_R1_C = sum_N2_R1_C + N2_R1_C['spec_cube'][int(row),int(col),int(i)]
    vflux_N2_R1_C.append(sum_N2_R1_C)

slambda_N2_R1_C = []
sflux_N2_R1_C = []

for i in range(len(N2_R1_C['wavenumber'])):
    slambda_N2_R1_C.append(N2_R1_A['wavenumber'][i])
    sum_N2_R1_C=0
    for j in range(len(disk_N2_R1_C)): #disk[j,0]:
        row=disk_N2_R1_C[j,0]
        col=disk_N2_R1_C[j,1]
        sum_N2_R1_C = sum_N2_R1_C + N2_R1_C['sky'][int(row),int(col),int(i)]
    sflux_N2_R1_C.append(sum_N2_R1_C)

In [19]:
vlambda_N2_R1_tot = []
vflux_N2_R1_tot = []
for i in range(len(N2_R1_tot['wavenumber'])):
    vlambda_N2_R1_tot.append(N2_R1_tot['wavenumber'][i])
    sum_N2_R1_tot=0
    for j in range(len(disk_N2_R1_tot)): #disk[j,0]:
        row=disk_N2_R1_tot[j,0]
        col=disk_N2_R1_tot[j,1]
        sum_N2_R1_tot = sum_N2_R1_tot + N2_R1_tot['spec_cube'][int(row),int(col),int(i)]
    vflux_N2_R1_tot.append(sum_N2_R1_tot)

slambda_N2_R1_tot = []
sflux_N2_R1_tot = []

for i in range(len(N2_R1_tot['wavenumber'])):
    slambda_N2_R1_tot.append(N2_R1_tot['wavenumber'][i])
    sum_N2_R1_tot=0
    for j in range(len(disk_N2_R1_tot)): #disk[j,0]:
        row=disk_N2_R1_tot[j,0]
        col=disk_N2_R1_tot[j,1]
        sum_N2_R1_tot = sum_N2_R1_tot + N2_R1_tot['sky'][int(row),int(col),int(i)]
    sflux_N2_R1_tot.append(sum_N2_R1_tot)

### Region Two

In [20]:
disk_N1_R2_A =np.argwhere(N1_R2_A.airmass>0)
disk_N1_R2_B =np.argwhere(N1_R2_B.airmass>0)
disk_N1_R2_tot =np.argwhere(N1_R2_tot.airmass>0)

disk_N2_R2_A =np.argwhere(N2_R2_A.airmass>0)
disk_N2_R2_B =np.argwhere(N2_R2_B.airmass>0)
disk_N2_R2_C =np.argwhere(N2_R2_C.airmass>0)
disk_N2_R2_tot =np.argwhere(N2_R2_tot.airmass>0)

In [21]:
vlambda_N1_R2_A = []
vflux_N1_R2_A = []
for i in range(len(N1_R2_A['wavenumber'])):
    vlambda_N1_R2_A.append(N1_R2_A['wavenumber'][i])
    sum_N1_R2_A=0
    for j in range(len(disk_N1_R2_A)): #disk[j,0]:
        row=disk_N1_R2_A[j,0]
        col=disk_N1_R2_A[j,1]
        sum_N1_R2_A = sum_N1_R2_A + N1_R2_A['spec_cube'][int(row),int(col),int(i)]
    vflux_N1_R2_A.append(sum_N1_R2_A)

slambda_N1_R2_A = []
sflux_N1_R2_A = []

for i in range(len(N1_R2_A['wavenumber'])):
    slambda_N1_R2_A.append(N1_R2_A['wavenumber'][i])
    sum_N1_R2_A=0
    for j in range(len(disk_N1_R2_A)): #disk[j,0]:
        row=disk_N1_R2_A[j,0]
        col=disk_N1_R2_A[j,1]
        sum_N1_R2_A = sum_N1_R2_A + N1_R2_A['sky'][int(row),int(col),int(i)]
    sflux_N1_R2_A.append(sum_N1_R2_A)

In [22]:
vlambda_N1_R2_B = []
vflux_N1_R2_B = []
for i in range(len(N1_R2_B['wavenumber'])):
    vlambda_N1_R2_B.append(N1_R2_B['wavenumber'][i])
    sum_N1_R2_B=0
    for j in range(len(disk_N1_R2_B)): #disk[j,0]:
        row=disk_N1_R2_B[j,0]
        col=disk_N1_R2_B[j,1]
        sum_N1_R2_B = sum_N1_R2_B + N1_R2_B['spec_cube'][int(row),int(col),int(i)]
    vflux_N1_R2_B.append(sum_N1_R2_B)

slambda_N1_R2_B = []
sflux_N1_R2_B = []

for i in range(len(N1_R2_B['wavenumber'])):
    slambda_N1_R2_B.append(N1_R2_B['wavenumber'][i])
    sum_N1_R2_B=0
    for j in range(len(disk_N1_R2_B)): #disk[j,0]:
        row=disk_N1_R2_B[j,0]
        col=disk_N1_R2_B[j,1]
        sum_N1_R2_B = sum_N1_R2_B + N1_R2_B['sky'][int(row),int(col),int(i)]
    sflux_N1_R2_B.append(sum_N1_R2_B)

In [23]:
vlambda_N1_R2_tot = []
vflux_N1_R2_tot = []
for i in range(len(N1_R2_tot['wavenumber'])):
    vlambda_N1_R2_tot.append(N1_R2_tot['wavenumber'][i])
    sum_N1_R2_tot=0
    for j in range(len(disk_N1_R2_tot)): #disk[j,0]:
        row=disk_N1_R2_tot[j,0]
        col=disk_N1_R2_tot[j,1]
        sum_N1_R2_tot = sum_N1_R2_tot + N1_R2_tot['spec_cube'][int(row),int(col),int(i)]
    vflux_N1_R2_tot.append(sum_N1_R2_tot)

slambda_N1_R2_tot = []
sflux_N1_R2_tot = []

for i in range(len(N1_R2_tot['wavenumber'])):
    slambda_N1_R2_tot.append(N1_R2_tot['wavenumber'][i])
    sum_N1_R2_tot=0
    for j in range(len(disk_N1_R2_tot)): #disk[j,0]:
        row=disk_N1_R2_tot[j,0]
        col=disk_N1_R2_tot[j,1]
        sum_N1_R2_tot = sum_N1_R2_tot + N1_R2_tot['sky'][int(row),int(col),int(i)]
    sflux_N1_R2_tot.append(sum_N1_R2_tot)

In [24]:
vlambda_N2_R2_A = []
vflux_N2_R2_A = []
for i in range(len(N2_R2_A['wavenumber'])):
    vlambda_N2_R2_A.append(N2_R2_A['wavenumber'][i])
    sum_N2_R2_A=0
    for j in range(len(disk_N2_R2_A)): #disk[j,0]:
        row=disk_N2_R2_A[j,0]
        col=disk_N2_R2_A[j,1]
        sum_N2_R2_A = sum_N2_R2_A + N2_R2_A['spec_cube'][int(row),int(col),int(i)]
    vflux_N2_R2_A.append(sum_N2_R2_A)

slambda_N2_R2_A = []
sflux_N2_R2_A = []

for i in range(len(N2_R2_A['wavenumber'])):
    slambda_N2_R2_A.append(N2_R2_A['wavenumber'][i])
    sum_N2_R2_A=0
    for j in range(len(disk_N2_R2_A)): #disk[j,0]:
        row=disk_N2_R2_A[j,0]
        col=disk_N2_R2_A[j,1]
        sum_N2_R2_A = sum_N2_R2_A + N2_R2_A['sky'][int(row),int(col),int(i)]
    sflux_N2_R2_A.append(sum_N2_R2_A)

In [25]:
vlambda_N2_R2_B = []
vflux_N2_R2_B = []
for i in range(len(N2_R2_B['wavenumber'])):
    vlambda_N2_R2_B.append(N2_R2_B['wavenumber'][i])
    sum_N2_R2_B=0
    for j in range(len(disk_N2_R2_B)): #disk[j,0]:
        row=disk_N2_R2_B[j,0]
        col=disk_N2_R2_B[j,1]
        sum_N2_R2_B = sum_N2_R2_B + N2_R2_B['spec_cube'][int(row),int(col),int(i)]
    vflux_N2_R2_B.append(sum_N2_R2_B)

slambda_N2_R2_B = []
sflux_N2_R2_B = []

for i in range(len(N2_R2_B['wavenumber'])):
    slambda_N2_R2_B.append(N2_R2_B['wavenumber'][i])
    sum_N2_R2_B=0
    for j in range(len(disk_N2_R2_B)): #disk[j,0]:
        row=disk_N2_R2_B[j,0]
        col=disk_N2_R2_B[j,1]
        sum_N2_R2_B = sum_N2_R2_B + N2_R2_B['sky'][int(row),int(col),int(i)]
    sflux_N2_R2_B.append(sum_N2_R2_B)

In [26]:
vlambda_N2_R2_C = []
vflux_N2_R2_C = []
for i in range(len(N2_R2_C['wavenumber'])):
    vlambda_N2_R2_C.append(N2_R2_C['wavenumber'][i])
    sum_N2_R2_C=0
    for j in range(len(disk_N2_R2_C)): #disk[j,0]:
        row=disk_N2_R2_C[j,0]
        col=disk_N2_R2_C[j,1]
        sum_N2_R2_C = sum_N2_R2_C + N2_R2_C['spec_cube'][int(row),int(col),int(i)]
    vflux_N2_R2_C.append(sum_N2_R2_C)

slambda_N2_R2_C = []
sflux_N2_R2_C = []

for i in range(len(N2_R2_C['wavenumber'])):
    slambda_N2_R2_C.append(N2_R2_A['wavenumber'][i])
    sum_N2_R2_C=0
    for j in range(len(disk_N2_R2_C)): #disk[j,0]:
        row=disk_N2_R2_C[j,0]
        col=disk_N2_R2_C[j,1]
        sum_N2_R2_C = sum_N2_R2_C + N2_R2_C['sky'][int(row),int(col),int(i)]
    sflux_N2_R2_C.append(sum_N2_R2_C)

In [27]:
vlambda_N2_R2_tot = []
vflux_N2_R2_tot = []
for i in range(len(N2_R2_tot['wavenumber'])):
    vlambda_N2_R2_tot.append(N2_R2_tot['wavenumber'][i])
    sum_N2_R2_tot=0
    for j in range(len(disk_N2_R2_tot)): #disk[j,0]:
        row=disk_N2_R2_tot[j,0]
        col=disk_N2_R2_tot[j,1]
        sum_N2_R2_tot = sum_N2_R2_tot + N2_R2_tot['spec_cube'][int(row),int(col),int(i)]
    vflux_N2_R2_tot.append(sum_N2_R2_tot)

slambda_N2_R2_tot = []
sflux_N2_R2_tot = []

for i in range(len(N2_R2_tot['wavenumber'])):
    slambda_N2_R2_tot.append(N2_R2_tot['wavenumber'][i])
    sum_N2_R2_tot=0
    for j in range(len(disk_N2_R2_tot)): #disk[j,0]:
        row=disk_N2_R2_tot[j,0]
        col=disk_N2_R2_tot[j,1]
        sum_N2_R2_tot = sum_N2_R2_tot + N2_R2_tot['sky'][int(row),int(col),int(i)]
    sflux_N2_R2_tot.append(sum_N2_R2_tot)

# Plotting

## Ranges

In [28]:
vrange_N1_R1_A = (np.max(vflux_N1_R1_A))-(np.min(vflux_N1_R1_A))
print(vrange_N1_R1_A)
print(np.max(vflux_N1_R1_A))
print(np.min(vflux_N1_R2_tot))
srange_N1_R1_A = (np.max(sflux_N1_R1_A))-(np.min(sflux_N1_R1_A))

vrange_N1_R1_B = (np.max(vflux_N1_R1_B))-(np.min(vflux_N1_R1_B))
srange_N1_R1_B = (np.max(sflux_N1_R1_B))-(np.min(sflux_N1_R1_B))

vrange_N1_R1_C = (np.max(vflux_N1_R1_C))-(np.min(vflux_N1_R1_C))
srange_N1_R1_C = (np.max(sflux_N1_R1_C))-(np.min(sflux_N1_R1_C))

vrange_N1_R1_tot = (np.max(vflux_N1_R1_tot))-(np.min(vflux_N1_R1_tot))
srange_N1_R1_tot = (np.max(sflux_N1_R1_tot))-(np.min(sflux_N1_R1_tot))

5098.155027389526
5098.155027389526
0.0


In [29]:
vrange_N2_R1_A = (np.max(vflux_N1_R1_A))-(np.min(vflux_N1_R1_A))
srange_N2_R1_A = (np.max(sflux_N1_R1_A))-(np.min(sflux_N1_R1_A))

vrange_N2_R1_B = (np.max(vflux_N1_R1_B))-(np.min(vflux_N1_R1_B))
srange_N2_R1_B = (np.max(sflux_N1_R1_B))-(np.min(sflux_N1_R1_B))

vrange_N2_R1_C = (np.max(vflux_N1_R1_C))-(np.min(vflux_N1_R1_C))
srange_N2_R1_C = (np.max(sflux_N1_R1_C))-(np.min(sflux_N1_R1_C))

vrange_N2_R1_tot = (np.max(vflux_N1_R1_tot))-(np.min(vflux_N1_R1_tot))
srange_N2_R1_tot = (np.max(sflux_N1_R1_tot))-(np.min(sflux_N1_R1_tot))

In [30]:
vrange_N1_R2_A = (np.max(vflux_N1_R1_A))-(np.min(vflux_N1_R1_A))
srange_N1_R2_A = (np.max(sflux_N1_R1_A))-(np.min(sflux_N1_R1_A))

vrange_N1_R2_B = (np.max(vflux_N1_R1_B))-(np.min(vflux_N1_R1_B))
srange_N1_R2_B = (np.max(sflux_N1_R1_B))-(np.min(sflux_N1_R1_B))

vrange_N1_R2_tot = (np.max(vflux_N1_R1_tot))-(np.min(vflux_N1_R1_tot))
srange_N1_R2_tot = (np.max(sflux_N1_R1_tot))-(np.min(sflux_N1_R1_tot))

In [31]:
vrange_N2_R2_A = (np.max(vflux_N1_R1_A))-(np.min(vflux_N1_R1_A))
srange_N2_R2_A = (np.max(sflux_N1_R1_A))-(np.min(sflux_N1_R1_A))

vrange_N2_R2_B = (np.max(vflux_N1_R1_B))-(np.min(vflux_N1_R1_B))
srange_N2_R2_B = (np.max(sflux_N1_R1_B))-(np.min(sflux_N1_R1_B))

vrange_N2_R2_C = (np.max(vflux_N1_R1_C))-(np.min(vflux_N1_R1_C))
srange_N2_R2_C = (np.max(sflux_N1_R1_C))-(np.min(sflux_N1_R1_C))

vrange_N2_R2_tot = (np.max(vflux_N1_R1_tot))-(np.min(vflux_N1_R1_tot))
srange_N2_R2_tot = (np.max(sflux_N1_R1_tot))-(np.min(sflux_N1_R1_tot))

## Night One Region One

In [32]:
TOOLTIPS = [ ("x","$x{‘00.00000000’}"),("y","$y{‘00.000’}"), ("strength", "$name")]

q = figure(title="Night One Region One A",sizing_mode="stretch_width",
    height=550,tooltips=TOOLTIPS,
    x_axis_label="Wavenumber (cm⁻¹)",
    y_axis_label="Radiance (erg·s⁻¹·cm⁻²·sr⁻¹)")


q.line(vlambda_N1_R1_A,-.01+(((vflux_N1_R1_A-(np.min(vflux_N1_R1_A)))*(1+.01))/ vrange_N1_R1_A),legend_label='Venus N1 R1 A', color='red')
#q.line(vlambda_N1_R1_B,-.01+(((vflux_N1_R1_B-(np.min(vflux_N1_R1_B)))*(1+.01))/ vrange_N1_R1_B),legend_label='Venus N1 R1 B', color='red')
#q.line(vlambda_N1_R1_C,-.01+(((vflux_N1_R1_C-(np.min(vflux_N1_R1_C)))*(1+.01))/ vrange_N1_R1_C),legend_label='Venus N1 R1 C', color='red')
#q.line(vlambda_N1_R1_tot,-.01+(((vflux_N1_R1_tot-(np.min(vflux_N1_R1_tot)))*(1+.01))/ vrange_N1_R1_tot),legend_label='Venus N1 R1 Total', color='red')

q.line(slambda_N1_R1_A,-.01+(((sflux_N1_R1_A-(np.min(sflux_N1_R1_A)))*(1+.01))/ srange_N1_R1_A),legend_label='Sky N1 R1 A', color='blue')
#q.line(slambda_N1_R1_B,-.01+(((sflux_N1_R1_B-(np.min(sflux_N1_R1_B)))*(1+.01))/ srange_N1_R1_B),legend_label='Sky N1 R1 B', color='blue')
#q.line(slambda_N1_R1_C,-.01+(((sflux_N1_R1_C-(np.min(sflux_N1_R1_C)))*(1+.01))/ srange_N1_R1_C),legend_label='Sky N1 R1 C', color='blue')
#q.line(slambda_N1_R1_tot,-.01+(((sflux_N1_R1_tot-(np.min(sflux_N1_R1_tot)))*(1+.01))/ srange_N1_R1_tot),legend_label='Sky N1 R1 Total', color='blue')


q.legend.location = "bottom_right"

show(q)

## Night Two Region One

In [33]:
TOOLTIPS = [ ("x","$x{‘00.00000000’}"),("y","$y{‘00.000’}"), ("strength", "$name")]

q = figure(title="Night Two Region One",sizing_mode="stretch_width",
    height=550,tooltips=TOOLTIPS,
    x_axis_label="Wavenumber (cm⁻¹)",
    y_axis_label="Radiance (erg·s⁻¹·cm⁻²·sr⁻¹)")


#q.line(vlambda_N2_R1_A,-.01+(((vflux_N2_R1_A-(np.min(vflux_N2_R1_A)))*(1+.01))/ vrange_N2_R1_A),legend_label='Venus N2 R1 A', color='red')
#q.line(vlambda_N2_R1_B,-.01+(((vflux_N2_R1_B-(np.min(vflux_N2_R1_B)))*(1+.01))/ vrange_N2_R1_B),legend_label='Venus N2 R1 B', color='red')
#q.line(vlambda_N2_R1_C,-.01+(((vflux_N2_R1_C-(np.min(vflux_N2_R1_C)))*(1+.01))/ vrange_N2_R1_C),legend_label='Venus N2 R1 C', color='red')
q.line(vlambda_N2_R1_tot,-.01+(((vflux_N2_R1_tot-(np.min(vflux_N2_R1_tot)))*(1+.01))/ vrange_N2_R1_tot),legend_label='Venus N2 R1 Total', color='red')

#q.line(slambda_N2_R1_A,-.01+(((sflux_N2_R1_A-(np.min(sflux_N2_R1_A)))*(1+.01))/ srange_N2_R1_A),legend_label='Sky N2 R1 A', color='blue')
#q.line(slambda_N2_R1_B,-.01+(((sflux_N2_R1_B-(np.min(sflux_N2_R1_B)))*(1+.01))/ srange_N2_R1_B),legend_label='Sky N2 R1 B', color='blue')
#q.line(slambda_N2_R1_C,-.01+(((sflux_N2_R1_C-(np.min(sflux_N2_R1_C)))*(1+.01))/ srange_N2_R1_C),legend_label='Sky N2 R1 C', color='blue')
q.line(slambda_N2_R1_tot,-.01+(((sflux_N2_R1_tot-(np.min(sflux_N2_R1_tot)))*(1+.01))/ srange_N2_R1_tot),legend_label='Sky N2 R1 Total', color='blue')


q.legend.location = "bottom_right"

show(q)

## Night One Region Two

In [124]:
TOOLTIPS = [ ("x","$x{‘00.00000000’}"),("y","$y{‘00.000’}"), ("strength", "$name")]

q = figure(title="2025B - Sky vs. Unprocessed",sizing_mode="stretch_width",
    height=550,tooltips=TOOLTIPS,
    x_axis_label="Wavenumber (cm⁻¹)",
    y_axis_label="Intensity")


#q.line(vlambda_N1_R2_A,-.01+(((vflux_N1_R2_A-(np.min(vflux_N1_R2_A)))*(1+.01))/ vrange_N1_R2_A),legend_label='Venus N1 R2 A', color='red')
#q.line(vlambda_N1_R2_B,-.01+(((vflux_N1_R2_B-(np.min(vflux_N1_R2_B)))*(1+.01))/ vrange_N1_R2_B),legend_label='Venus N1 R2 B', color='red')
q.line(vlambda_N1_R2_tot,-.01+(((vflux_N1_R2_tot-(np.min(vflux_N1_R2_tot)))*(1+.01))/ vrange_N1_R2_tot),legend_label='2025B - Unprocessed', color='green')

#q.line(slambda_N1_R2_A,-.01+(((sflux_N1_R2_A-(np.min(sflux_N1_R2_A)))*(1+.01))/ srange_N1_R2_A),legend_label='Sky N1 R2 A', color='blue')
#q.line(slambda_N1_R2_B,-.01+(((sflux_N1_R2_B-(np.min(sflux_N1_R2_B)))*(1+.01))/ srange_N1_R2_B),legend_label='Sky N1 R2 B', color='blue')
q.line(slambda_N1_R2_tot,-.01+(((sflux_N1_R2_tot-(np.min(sflux_N1_R2_tot)))*(1+.01))/ srange_N1_R2_tot),legend_label='2025B - Sky', color='blue')

q.y_range = Range1d(0.52, 0.9)

q.legend.location = "bottom_right"

show(q)

## Night Two Region Two

In [35]:
TOOLTIPS = [ ("x","$x{‘00.00000000’}"),("y","$y{‘00.000’}"), ("strength", "$name")]

q = figure(title="Night Two Region Two",sizing_mode="stretch_width",
    height=550,tooltips=TOOLTIPS,
    x_axis_label="Wavenumber (cm⁻¹)",
    y_axis_label="Radiance (erg·s⁻¹·cm⁻²·sr⁻¹)")


#q.line(vlambda_N2_R2_A,-.01+(((vflux_N2_R2_A-(np.min(vflux_N2_R2_A)))*(1+.01))/ vrange_N2_R2_A),legend_label='Venus N2 R2 A', color='red')
#q.line(vlambda_N2_R2_B,-.01+(((vflux_N2_R2_B-(np.min(vflux_N2_R2_B)))*(1+.01))/ vrange_N2_R2_B),legend_label='Venus N2 R2 B', color='red')
#q.line(vlambda_N2_R2_C,-.01+(((vflux_N2_R2_C-(np.min(vflux_N2_R2_C)))*(1+.01))/ vrange_N2_R2_C),legend_label='Venus N2 R2 C', color='red')
q.line(vlambda_N2_R2_tot,-.01+(((vflux_N2_R2_tot-(np.min(vflux_N2_R2_tot)))*(1+.01))/ vrange_N2_R2_tot),legend_label='Venus N2 R2 Total', color='red')

#q.line(slambda_N2_R2_A,-.01+(((sflux_N2_R2_A-(np.min(sflux_N2_R2_A)))*(1+.01))/ srange_N2_R2_A),legend_label='Sky N2 R2 A', color='blue')
#q.line(slambda_N2_R2_B,-.01+(((sflux_N2_R2_B-(np.min(sflux_N2_R2_B)))*(1+.01))/ srange_N2_R2_B),legend_label='Sky N2 R2 B', color='blue')
#q.line(slambda_N2_R2_C,-.01+(((sflux_N2_R2_C-(np.min(sflux_N2_R2_C)))*(1+.01))/ srange_N2_R2_C),legend_label='Sky N2 R2 C', color='blue')
q.line(slambda_N2_R2_tot,-.01+(((sflux_N2_R2_tot-(np.min(sflux_N2_R2_tot)))*(1+.01))/ srange_N2_R2_tot),legend_label='Sky N2 R2 Total', color='blue')


q.legend.location = "bottom_right"

show(q)

# PSG Spectra

In [36]:
#setting variables, one is the PSG wavenumber, one is the composite spectrum, and the rest are the individual spectra that make up PSG's composite spectrum
R1_rev = []
R1_tot_rev = []
R1_H2O_rev = []
R1_CO2_rev = []
R1_O3_rev = []
R1_N2O_rev = []
R1_CO_rev = []
R1_CH4_rev = []
R1_O2_rev = []
R1_N2_rev = []
R1_Rayleigh_rev = []
R1_CIA_rev = []

#defining each variable with a row contained in the PSG .csv file
with open('/Users/physicsstudent2/desktop/venus_research/BSRI_2024/PSG/R1_PSG.csv') as r1_psg_file:  
    plots = csv.reader(r1_psg_file, delimiter = ',') 
      
    for row in plots: 
        R1_rev.append(float(row[0]))
        R1_tot_rev.append(float(row[1]))
        R1_H2O_rev.append(float(row[2]))
        R1_CO2_rev.append(float(row[3]))
        R1_O3_rev.append(float(row[4]))
        R1_N2O_rev.append(float(row[5]))
        R1_CO_rev.append(float(row[6]))
        R1_CH4_rev.append(float(row[7]))
        R1_O2_rev.append(float(row[8]))
        R1_N2_rev.append(float(row[9]))
        R1_Rayleigh_rev.append(float(row[10]))
        R1_CIA_rev.append(float(row[11]))

In [37]:
TOOLTIPS = [ ("x","$x{‘00.00000000’}"),("y","$y{‘00.000’}"), ("strength", "$name")]
q = figure(title="Region One PSG",sizing_mode="stretch_width",
    height=500,tooltips=TOOLTIPS,
    x_axis_label="Wavenumber (cm⁻¹)",
    y_axis_label="Radiance (erg·s⁻¹·cm⁻²·sr⁻¹)")

q.line(R1_rev, R1_tot_rev, legend_label='Earth', color='thistle') #consists of H2O, O3, N2O, and CH4
q.line(R1_rev, R1_H2O_rev,legend_label='H2O', color='navy') #three features present
q.line(R1_rev, R1_CO2_rev,legend_label='CO2', color='navajowhite')
q.line(R1_rev, R1_O3_rev,legend_label='O3', color='darkseagreen') #primariliy present
q.line(R1_rev, R1_N2O_rev,legend_label='N2O', color='lavender') #largely present
q.line(R1_rev, R1_CO_rev,legend_label='CO', color='khaki') 
q.line(R1_rev, R1_CH4_rev,legend_label='CH4', color='lightcoral') #present 
q.line(R1_rev, R1_O2_rev,legend_label='O2', color='lightskyblue')
q.line(R1_rev, R1_N2_rev,legend_label='N2', color='mediumseagreen')
q.line(R1_rev, R1_Rayleigh_rev,legend_label='Rayleigh', color='rosybrown')
q.line(R1_rev, R1_CIA_rev,legend_label='CIA', color='burlywood')


q.legend.location = "bottom_right"

show(q)

In [38]:
#setting variables, one is the PSG wavenumber, one is the composite spectrum, and the rest are the individual spectra that make up PSG's composite spectrum
R2_rev = []
R2_tot_rev = []
R2_H2O_rev = []
R2_CO2_rev = []
R2_O3_rev = []
R2_N2O_rev = []
R2_CO_rev = []
R2_CH4_rev = []
R2_O2_rev = []
R2_N2_rev = []
R2_Rayleigh_rev = []
R2_CIA_rev = []

#defining each variable with a row contained in the PSG .csv file
with open('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/second_wn/tell_2025.csv') as r2_psg_file:  
    plots = csv.reader(r2_psg_file, delimiter = ',') 
      
    for row in plots: 
        R2_rev.append(float(row[0]))
        R2_tot_rev.append(float(row[1]))
        R2_H2O_rev.append(float(row[2]))
        R2_CO2_rev.append(float(row[3]))
        R2_O3_rev.append(float(row[4]))
        R2_N2O_rev.append(float(row[5]))
        R2_CO_rev.append(float(row[6]))
        R2_CH4_rev.append(float(row[7]))
        R2_O2_rev.append(float(row[8]))
        R2_N2_rev.append(float(row[9]))
        R2_Rayleigh_rev.append(float(row[10]))
        R2_CIA_rev.append(float(row[11]))

In [125]:
TOOLTIPS = [ ("x","$x{‘00.00000000’}"),("y","$y{‘00.000’}"), ("strength", "$name")]
q = figure(title="Wavenumber Region for 2025B Telluric PSG",sizing_mode="stretch_width",
    height=500,tooltips=TOOLTIPS,
    x_axis_label="Wavenumber (cm⁻¹)",
    y_axis_label="Intensity",
    x_range = (1204,1209))

q.line(R2_rev, R2_tot_rev, legend_label='Earth',line_width=1.5, color='black') #consists of H2O, O3, N2O, and CH4
q.line(R2_rev, R2_H2O_rev,legend_label='H2O', color='blue') #three features present
q.line(R2_rev, R2_O3_rev,legend_label='O3', color='green') #primariliy present
q.line(R2_rev, R2_CH4_rev,legend_label='CH4', color='red') #present 
q.line(R2_rev, R2_N2O_rev,legend_label='N2O', color='orange') #largely present
q.line(R2_rev, R2_CO2_rev,legend_label='CO2', color='gray')
q.line(R2_rev, R2_CO_rev,legend_label='CO', color='gray') 
q.line(R2_rev, R2_O2_rev,legend_label='O2', color='gray')
q.line(R2_rev, R2_N2_rev,legend_label='N2', color='gray')
q.line(R2_rev, R2_Rayleigh_rev,legend_label='Rayleigh', color='gray')
q.line(R2_rev, R2_CIA_rev,legend_label='CIA', color='gray')



q.legend.location = "bottom_right"

show(q)

# Data Processing

## Variables

In [40]:
R1_lambda = list(reversed(R1_rev))
R1_lambdads = []
for number in R1_lambda:
    R1_lambdads.append(number+0.0315159369) #Plus the Doppler Shift

R1_H2O = list(reversed(R1_H2O_rev))
R1_O3 = list(reversed(R1_O3_rev))
R1_N2O = list(reversed(R1_N2O_rev))
R1_CH4 = list(reversed(R1_CH4_rev))

psg_step_size = ((max(R1_lambdads)-min(R1_lambdads))/len(R1_lambdads))
print(psg_step_size)
step_N1_R1_tot = ((max(vlambda_N1_R1_tot)-min(vlambda_N1_R1_tot))/len(vlambda_N1_R1_tot))
print(step_N1_R1_tot)

0.0022354835680750956
0.0037142318573480885


In [41]:
R2_lambda = list(reversed(R2_rev))
R2_lambdads = []
for number in R2_lambda:
    R2_lambdads.append(number+0.0315159369) #Plus the Doppler Shift on N1 of Observation

R2_H2O = list(reversed(R2_H2O_rev))
R2_O3 = list(reversed(R2_O3_rev))
R2_N2O = list(reversed(R2_N2O_rev))
R2_CH4 = list(reversed(R2_CH4_rev))

psg2_step_size = ((max(R2_lambdads)-min(R2_lambdads))/len(R2_lambdads))
print(psg_step_size)
step_N1_R2_tot = ((max(vlambda_N1_R2_tot)-min(vlambda_N1_R2_tot))/len(vlambda_N1_R2_tot))
print(step_N1_R1_tot)

0.0022354835680750956
0.0037142318573480885


## Night One Region One Total

In [42]:
vnorm_N1_R1_tot = -.01+(((vflux_N1_R1_tot-(np.min(vflux_N1_R1_tot)))*(1+.01))/ vrange_N1_R1_tot)
snorm_N1_R1_tot = -.01+(((sflux_N1_R1_tot-(np.min(sflux_N1_R1_tot)))*(1+.01))/ srange_N1_R1_tot)

In [43]:
print(min(R1_H2O))
print(R1_H2O.index(0.057861))
print(R1_lambda[3683])
print(min(vnorm_N1_R1_tot[1900:2000]))
print(np.where(vnorm_N1_R1_tot == 0.30934588450901235))
print(vlambda_N1_R1_tot[1967])

#print(vlambda_N1_R1_tot[1900:2000]) 
print(len(vnorm_N1_R1_tot))
print(len(vflux_N2_R2_tot))

H2Oexp_N1_R1_tot = np.log(min(vnorm_N1_R1_tot[1900:2000]))/np.log(min(R1_H2O)) + 0.1
print(H2Oexp_N1_R1_tot)
print(vlambda_N1_R1_tot[1967]-R1_lambda[3683])

0.057861
3683
1121.229305
0.30934588450901235
(array([1967]),)
1121.2655949026803
2385
1732
0.5117241972977327
0.03628990268020971


In [44]:
print(min(R1_O3))
print(R1_O3.index(0.0513534))
print(R1_lambda[3475])

#print(vlambda_N1_R1_tot[1800:1900])
print(min(vnorm_N1_R1_tot[1800:1900]))

#O3exp_N1_R1_tot = np.log(min(vnorm_N1_R1_tot[1800:1900]))/np.log(min(R1_O3)) + 0.4
O3exp_N1_R1_tot = 0.68
print(O3exp_N1_R1_tot)

0.0513534
3475
1120.762971
0.4810451686350986
0.68


In [45]:
print(min(R1_N2O))
print(R1_N2O.index(0.999004))
print(R1_lambda[4362])

#print(vlambda_N1_R1_tot[2050:2150])
print(min(vnorm_N1_R1_tot[2050:2150]))

#N2Oexp_N1_R1_tot = np.log(min(vnorm_N1_R1_tot[2050:2150]))/np.log(min(R1_N2O))
N2Oexp_N1_R1_tot = 0.1
print(N2Oexp_N1_R1_tot)

0.999004
4362
1122.752967
0.6561369092180316
0.1


In [46]:
print(min(R1_CH4))
print(R1_CH4.index(0.992888))
print(R1_lambda[4194])

#print(vlambda_N1_R1_tot[1950:2050])
print(min(vnorm_N1_R1_tot[1950:2050]))

#CH4exp_N1_R1_tot = np.log(min(vnorm_N1_R1_tot[2050:2150]))/np.log(min(R1_CH4))
CH4exp_N1_R1_tot = 0.1
print(CH4exp_N1_R1_tot)

0.992888
4194
1122.375786
0.30934588450901235
0.1


In [47]:
k_N1_R1_tot = 2 #Gaussian sigma of resolution broadening, in pixels
lambshift_N1_R1_tot = 0 #empirical wavenumber shift to improve residuals

In [48]:
#multiplying the spectra by their respective exponents
H2O_N1_R1_tot=[]
O3_N1_R1_tot=[]
N2O_N1_R1_tot=[]
CH4_N1_R1_tot=[]
for number in R1_H2O:
    H2O_N1_R1_tot.append(number**H2Oexp_N1_R1_tot)
for number in R1_O3:
    O3_N1_R1_tot.append(number**O3exp_N1_R1_tot)
for number in R1_N2O:
    N2O_N1_R1_tot.append(number**N2Oexp_N1_R1_tot)
for number in R1_CH4:
    CH4_N1_R1_tot.append(number**CH4exp_N1_R1_tot)

#multiplying the exponentiated spectra together to create a new composite spectrum
tot_N1_R1_tot = np.array(H2O_N1_R1_tot)*np.array(O3_N1_R1_tot)*np.array(N2O_N1_R1_tot)*np.array(CH4_N1_R1_tot)

#gaussian smoothing the spectrum
gaus_N1_R1_tot = gaussian_filter(tot_N1_R1_tot,k_N1_R1_tot)

#interpolating the spectrum over the wavenumber range
int_N1_R1_tot = scipy.interpolate.interp1d(R1_lambdads, gaus_N1_R1_tot)
intfunc_N1_R1_tot = int_N1_R1_tot(vlambda_N1_R1_tot)
tell_N1_R1_tot = shift(intfunc_N1_R1_tot, lambshift_N1_R1_tot)

#normalizing the model
normtell_N1_R1_tot  = tell_N1_R1_tot / np.median(tell_N1_R1_tot)

#dividing the spectrum by the model
vproc_N1_R1_tot = np.divide(vnorm_N1_R1_tot, normtell_N1_R1_tot)

In [111]:
#plotting the processed file
TOOLTIPS = [ ("x","$x{‘00.00000000’}"),("y","$y{‘00.000’}"), ("strength", "$name")]
q = figure(title="2025A - Sky-Divided vs. Unprocessed",sizing_mode="stretch_width",
    height=500,tooltips=TOOLTIPS,
    x_axis_label="Wavenumber (cm⁻¹)",
    y_axis_label="Intensity")

#q.line(vlambda_N1_R1_tot, normtell_N1_R1_tot -0.1, legend_label='2025A - Corrected Telluric', color='orange')
q.line(vlambda_N1_R1_tot, vproc_N1_R1_tot, legend_label='2025A - Sky Divided', color='green')
q.line(vlambda_N1_R1_tot, vnorm_N1_R1_tot - 0.05, legend_label='2025A - Unprocessed', color='red')
#q.line(slambda_N1_R1_tot, snorm_N1_R1_tot,legend_label='sky', color='blue')

#for place in range(0,len(line_locations_region_1)):
    #q.line([range1_array[place,0]+0.03,range1_array[place,0]+0.03], [1.06,0.60],color='black',line_width=1,name=str(range1_array[place,1]), legend_label='Strong Telluric Features That Cannot Be Fully Processed')
#for place in range(0,len(two_line_locations_region_1)):
    #q.line([two_range1_array[place,0]+0.03,two_range1_array[place,0]+0.03], [1.06,0.60],color='black',line_width=1,name=str(two_range1_array[place,1]))

#q.x_range = Range1d(1116.95, 1117.04)
q.y_range = Range1d(0.25, 1)
    
q.legend.location = "bottom_right"

show(q)

## Night Two Region One Total

In [50]:
vnorm_N2_R1_tot = -.01+(((vflux_N2_R1_tot-(np.min(vflux_N2_R1_tot)))*(1+.01))/ vrange_N2_R1_tot)
snorm_N2_R1_tot = -.01+(((sflux_N2_R1_tot-(np.min(sflux_N2_R1_tot)))*(1+.01))/ srange_N2_R1_tot)

In [51]:
print(min(R1_H2O))
print(R1_H2O.index(0.057861))
print(R1_lambda[3683])

#print(vlambda_N2_R1_tot[1700:1800])
print(min(vnorm_N2_R1_tot[1700:1800]))

H2Oexp_N2_R1_tot = np.log(min(vnorm_N2_R1_tot[1700:1800]))/np.log(min(R1_H2O))
#H2Oexp_N2_R1_tot = 0.1
print(H2Oexp_N2_R1_tot)

0.057861
3683
1121.229305
0.25076819295628344
0.4853916634145342


In [52]:
print(min(R1_O3))
print(R1_O3.index(0.0513534))
print(R1_lambda[3475])

#print(vlambda_N2_R1_tot[1650:1750])
print(min(vnorm_N2_R1_tot[1650:1750]))

#O3exp_N2_R1_tot = np.log(min(vnorm_N2_R1_tot[1800:1900]))/np.log(min(R1_O3)) 
O3exp_N2_R1_tot = 0.65
print(O3exp_N2_R1_tot)

0.0513534
3475
1120.762971
0.4469444821105192
0.65


In [53]:
print(min(R1_N2O))
print(R1_N2O.index(0.999004))
print(R1_lambda[4362])

#print(vlambda_N2_R1_tot[2050:2150])
#print(min(vnorm_N2_R1_tot[2050:2150]))

#N2Oexp_N2_R1_tot = np.log(min(vnorm_N2_R1_tot[2050:2150]))/np.log(min(R1_N2O))
N2Oexp_N2_R1_tot = 0.1
print(N2Oexp_N2_R1_tot)

0.999004
4362
1122.752967
0.1


In [54]:
print(min(R1_CH4))
print(R1_CH4.index(0.992888))
print(R1_lambda[4194])

#print(vlambda_N2_R1_tot[2000:2050])
#print(min(vnorm_N2_R1_tot[2000:2100]))

#CH4exp_N2_R1_tot = np.log(min(vnorm_N2_R1_tot[2000:2100]))/np.log(min(R1_CH4))
CH4exp_N2_R1_tot = 0.1
print(CH4exp_N2_R1_tot)

0.992888
4194
1122.375786
0.1


In [55]:
k_N2_R1_tot = 2 #Gaussian sigma of resolution broadening, in pixels
lambshift_N2_R1_tot = -0.2 #empirical wavenumber shift to improve residuals

In [56]:
#multiplying the spectra by their respective exponents
H2O_N2_R1_tot=[]
O3_N2_R1_tot=[]
N2O_N2_R1_tot=[]
CH4_N2_R1_tot=[]
for number in R1_H2O:
    H2O_N2_R1_tot.append(number**H2Oexp_N2_R1_tot)
for number in R1_O3:
    O3_N2_R1_tot.append(number**O3exp_N2_R1_tot)
for number in R1_N2O:
    N2O_N2_R1_tot.append(number**N2Oexp_N2_R1_tot)
for number in R1_CH4:
    CH4_N2_R1_tot.append(number**CH4exp_N2_R1_tot)

#multiplying the exponentiated spectra together to create a new composite spectrum
tot_N2_R1_tot = np.array(H2O_N2_R1_tot)*np.array(O3_N2_R1_tot)*np.array(N2O_N2_R1_tot)*np.array(CH4_N2_R1_tot)

#gaussian smoothing the spectrum
gaus_N2_R1_tot = gaussian_filter(tot_N2_R1_tot,k_N2_R1_tot)

#interpolating the spectrum over the wavenumber range
int_N2_R1_tot = scipy.interpolate.interp1d(R1_lambdads, gaus_N2_R1_tot)
intfunc_N2_R1_tot = int_N2_R1_tot(vlambda_N2_R1_tot)
tell_N2_R1_tot = shift(intfunc_N2_R1_tot, lambshift_N2_R1_tot)

#normalizing the model
normtell_N2_R1_tot  = tell_N2_R1_tot / np.median(tell_N2_R1_tot)

#dividing the spectrum by the model
vproc_N2_R1_tot = np.divide(vnorm_N2_R1_tot, normtell_N2_R1_tot)

/var/folders/0q/22yph8ws4w999mfg8_bqcg840000gn/T/ipykernel_2872/922299277.py:30: RuntimeWarning: divide by zero encountered in divide
  vproc_N2_R1_tot = np.divide(vnorm_N2_R1_tot, normtell_N2_R1_tot)


In [57]:
#interpolating the spectrum over the wavenumber range
int_N2_R1_tot = scipy.interpolate.interp1d(R1_lambdads, gaus_N2_R1_tot)
intfunc_N2_R1_tot = int_N2_R1_tot(vlambda_N2_R1_tot)
tell_N2_R1_tot = shift(intfunc_N2_R1_tot, lambshift_N2_R1_tot)

#normalizing the model
normtell_N2_R1_tot  = tell_N2_R1_tot / np.median(tell_N2_R1_tot)

#dividing the spectrum by the model
vproc_N2_R1_tot = np.divide(vnorm_N2_R1_tot, normtell_N2_R1_tot)

/var/folders/0q/22yph8ws4w999mfg8_bqcg840000gn/T/ipykernel_2872/917587174.py:10: RuntimeWarning: divide by zero encountered in divide
  vproc_N2_R1_tot = np.divide(vnorm_N2_R1_tot, normtell_N2_R1_tot)


In [58]:
#plotting the processed file
TOOLTIPS = [ ("x","$x{‘00.00000000’}"),("y","$y{‘00.000’}"), ("strength", "$name")]
q = figure(title="Night Two Wavenumber Region One Total Disk",sizing_mode="stretch_width",
    height=500,tooltips=TOOLTIPS,
    x_axis_label="Wavenumber (cm⁻¹)",
    y_axis_label="Radiance (erg·s⁻¹·cm⁻²·sr⁻¹)")

#q.line(vlambda_N2_R1_tot, normtell_N2_R1_tot -0.1, legend_label='corrected telluric', color='orange')
q.line(vlambda_N2_R1_tot, vproc_N2_R1_tot + 0.1, legend_label='venus processed', color='green')
q.line(vlambda_N2_R1_tot, vnorm_N2_R1_tot, legend_label='venus unprocessed', color='red')
#q.line(vlambda_N2_R1_tot, snorm_N2_R1_tot,legend_label='sky', color='blue')

#for place in range(0,len(line_locations_region_1)):
    #q.line([range1_array[place,0]+0.03,range1_array[place,0]+0.03], [1.06,0.60],color='black',line_width=1,name=str(range1_array[place,1]), legend_label='Strong Telluric Features That Cannot Be Fully Processed')
#for place in range(0,len(two_line_locations_region_1)):
    #q.line([two_range1_array[place,0]+0.03,two_range1_array[place,0]+0.03], [1.06,0.60],color='black',line_width=1,name=str(two_range1_array[place,1]))

q.legend.location = "bottom_right"

show(q)

## Night One Region One A

In [59]:
vnorm_N1_R1_A = -.01+(((vflux_N1_R1_A-(np.min(vflux_N1_R1_A)))*(1+.01))/ vrange_N1_R1_A)
snorm_N1_R1_A = -.01+(((sflux_N1_R1_A-(np.min(sflux_N1_R1_A)))*(1+.01))/ srange_N1_R1_A)

In [60]:
print(min(R1_H2O))
print(R1_H2O.index(0.057861))
print(R1_lambda[3683])

#print(vlambda_N1_R1_A[1900:2000])
print(min(vnorm_N1_R1_A[1900:2000]))

H2Oexp_N1_R1_A = np.log(min(vnorm_N1_R1_A[1900:2000]))/np.log(min(R1_H2O))
#H2Oexp_N1_R1_A = 0.1
print(H2Oexp_N1_R1_A)

0.057861
3683
1121.229305
0.35811745614013374
0.3603502269651862


In [61]:
print(min(R1_O3))
print(R1_O3.index(0.0513534))
print(R1_lambda[3475])

#print(vlambda_N1_R1_A[1800:1900])
print(min(vnorm_N1_R1_A[1800:1900]))

#O3exp_N1_R1_A = np.log(min(vnorm_N1_R1_A[1800:1900]))/np.log(min(R1_O3))
O3exp_N1_R1_A = 0.7
print(O3exp_N1_R1_A)

0.0513534
3475
1120.762971
0.4632817797294081
0.7


In [62]:
N2Oexp_N1_R1_A = 0.1
print(N2Oexp_N1_R1_A)

0.1


In [63]:
CH4exp_N1_R1_A = 0.1
print(CH4exp_N1_R1_A)

0.1


In [64]:
k_N1_R1_A = 1.6 #Gaussian sigma of resolution broadening, in pixels
lambshift_N1_R1_A = 0.008 #empirical wavenumber shift to improve residuals

In [65]:
#multiplying the spectra by their respective exponents
H2O_N1_R1_A=[]
O3_N1_R1_A=[]
N2O_N1_R1_A=[]
CH4_N1_R1_A=[]
for number in R1_H2O:
    H2O_N1_R1_A.append(number**H2Oexp_N1_R1_A)
for number in R1_O3:
    O3_N1_R1_A.append(number**O3exp_N1_R1_A)
for number in R1_N2O:
    N2O_N1_R1_A.append(number**N2Oexp_N1_R1_A)
for number in R1_CH4:
    CH4_N1_R1_A.append(number**CH4exp_N1_R1_A)

#multiplying the exponentiated spectra together to create a new composite spectrum
tot_N1_R1_A = np.array(H2O_N1_R1_A)*np.array(O3_N1_R1_A)*np.array(N2O_N1_R1_A)*np.array(CH4_N1_R1_A)
print(tot_N1_R1_A)
print(tot_N1_R1_tot)

#gaussian smoothing the spectrum
gaus_N1_R1_A = gaussian_filter(tot_N1_R1_A,k_N1_R1_A)
print(gaus_N1_R1_A)

#interpolating the spectrum over the wavenumber range
int_N1_R1_A = scipy.interpolate.interp1d(R1_lambdads, gaus_N1_R1_A)
intfunc_N1_R1_A = int_N1_R1_A(vlambda_N1_R1_A)
tell_N1_R1_A = shift(intfunc_N1_R1_A, lambshift_N1_R1_A)

#normalizing the model
normtell_N1_R1_A  = tell_N1_R1_A / np.median(tell_N1_R1_A)

#dividing the spectrum by the model
vproc_N1_R1_A = np.divide(vnorm_N1_R1_A, normtell_N1_R1_A)

[0.9985415  0.99846595 0.99852725 ... 0.98144479 0.97126578 0.94121052]
[0.99833641 0.99826369 0.99832403 ... 0.98176055 0.97186817 0.94264076]
[0.99851786 0.99852231 0.99853232 ... 0.97384182 0.96620024 0.96017059]


/var/folders/0q/22yph8ws4w999mfg8_bqcg840000gn/T/ipykernel_2872/665108497.py:33: RuntimeWarning: divide by zero encountered in divide
  vproc_N1_R1_A = np.divide(vnorm_N1_R1_A, normtell_N1_R1_A)


In [66]:
#plotting the processed file
TOOLTIPS = [ ("x","$x{‘00.00000000’}"),("y","$y{‘00.000’}"), ("strength", "$name")]
q = figure(title="Night One WnRegion One GeoRegion A",sizing_mode="stretch_width",
    height=500,tooltips=TOOLTIPS,
    x_axis_label="Wavenumber (cm⁻¹)",
    y_axis_label="Radiance (erg·s⁻¹·cm⁻²·sr⁻¹)")

#q.line(vlambda_N1_R1_A, normtell_N1_R1_A -0.1, legend_label='corrected telluric', color='orange')
q.line(vlambda_N1_R1_A, vproc_N1_R1_A + 0.1, legend_label='venus processed', color='green')
q.line(vlambda_N1_R1_A, vnorm_N1_R1_A, legend_label='venus unprocessed', color='red')
#q.line(vlambda_N1_R1_A, snorm_N1_R1_A,legend_label='sky', color='blue')

#for place in range(0,len(line_locations_region_1)):
    #q.line([range1_array[place,0]+0.03,range1_array[place,0]+0.03], [1.06,0.60],color='black',line_width=1,name=str(range1_array[place,1]), legend_label='Strong Telluric Features That Cannot Be Fully Processed')
#for place in range(0,len(two_line_locations_region_1)):
    #q.line([two_range1_array[place,0]+0.03,two_range1_array[place,0]+0.03], [1.06,0.60],color='black',line_width=1,name=str(two_range1_array[place,1]))

q.legend.location = "bottom_right"

show(q)

## Night One Region One B

In [67]:
vnorm_N1_R1_B = -.01+(((vflux_N1_R1_B-(np.min(vflux_N1_R1_B)))*(1+.01))/ vrange_N1_R1_B)
snorm_N1_R1_B = -.01+(((sflux_N1_R1_B-(np.min(sflux_N1_R1_B)))*(1+.01))/ srange_N1_R1_B)

In [68]:
print(min(R1_H2O))
print(R1_H2O.index(0.057861))
print(R1_lambda[3683])

#print(vlambda_N1_R1_B[1900:2000])
print(min(vnorm_N1_R1_B[1900:2000]))

H2Oexp_N1_R1_B = np.log(min(vnorm_N1_R1_B[1900:2000]))/np.log(min(R1_H2O))
#H2Oexp_N1_R1_B = 0.1
print(H2Oexp_N1_R1_B)

0.057861
3683
1121.229305
0.3042605748280589
0.4175407611057886


In [69]:
print(min(R1_O3))
print(R1_O3.index(0.0513534))
print(R1_lambda[3475])

#print(vlambda_N1_R1_B[1800:1900])
print(min(vnorm_N1_R1_B[1800:1900]))

#O3exp_N1_R1_B = np.log(min(vnorm_N1_R1_B[1800:1900]))/np.log(min(R1_O3))
O3exp_N1_R1_B = 0.58
print(O3exp_N1_R1_B)

0.0513534
3475
1120.762971
0.5202608492527685
0.58


In [70]:
N2Oexp_N1_R1_B = 0.1
print(N2Oexp_N1_R1_B)

0.1


In [71]:
CH4exp_N1_R1_B = 0.1
print(CH4exp_N1_R1_B)

0.1


In [72]:
k_N1_R1_B = 1.8 #Gaussian sigma of resolution broadening, in pixels
lambshift_N1_R1_B = 0.008 #empirical wavenumber shift to improve residuals"

In [73]:
#multiplying the spectra by their respective exponents
H2O_N1_R1_B=[]
O3_N1_R1_B=[]
N2O_N1_R1_B=[]
CH4_N1_R1_B=[]
for number in R1_H2O:
    H2O_N1_R1_B.append(number**H2Oexp_N1_R1_B)
for number in R1_O3:
    O3_N1_R1_B.append(number**O3exp_N1_R1_B)
for number in R1_N2O:
    N2O_N1_R1_B.append(number**N2Oexp_N1_R1_B)
for number in R1_CH4:
    CH4_N1_R1_B.append(number**CH4exp_N1_R1_B)

#multiplying the exponentiated spectra together to create a new composite spectrum
tot_N1_R1_B = np.array(H2O_N1_R1_B)*np.array(O3_N1_R1_B)*np.array(N2O_N1_R1_B)*np.array(CH4_N1_R1_B)

#gaussian smoothing the spectrum
gaus_N1_R1_B = gaussian_filter(tot_N1_R1_B,k_N1_R1_B)

#interpolating the spectrum over the wavenumber range
int_N1_R1_B = scipy.interpolate.interp1d(R1_lambdads, gaus_N1_R1_B)
intfunc_N1_R1_B = int_N1_R1_B(vlambda_N1_R1_B)
tell_N1_R1_B = shift(intfunc_N1_R1_B, lambshift_N1_R1_B)

#normalizing the model
normtell_N1_R1_B = tell_N1_R1_B / np.median(tell_N1_R1_B)

#dividing the spectrum by the model
vproc_N1_R1_B = np.divide(vnorm_N1_R1_B, normtell_N1_R1_B)

/var/folders/0q/22yph8ws4w999mfg8_bqcg840000gn/T/ipykernel_2872/4040423708.py:30: RuntimeWarning: divide by zero encountered in divide
  vproc_N1_R1_B = np.divide(vnorm_N1_R1_B, normtell_N1_R1_B)


In [74]:
#plotting the processed file
TOOLTIPS = [ ("x","$x{‘00.00000000’}"),("y","$y{‘00.000’}"), ("strength", "$name")]
q = figure(title="Night One WnRegion One GeoRegion B",sizing_mode="stretch_width",
    height=500,tooltips=TOOLTIPS,
    x_axis_label="Wavenumber (cm⁻¹)",
    y_axis_label="Radiance (erg·s⁻¹·cm⁻²·sr⁻¹)")

#q.line(vlambda_N1_R1_B, normtell_N1_R1_B -0.1, legend_label='corrected telluric', color='orange')
q.line(vlambda_N1_R1_B, vproc_N1_R1_B + 0.1, legend_label='venus processed', color='green')
q.line(vlambda_N1_R1_B, vnorm_N1_R1_B, legend_label='venus unprocessed', color='red')
#q.line(vlambda_N1_R1_B, snorm_N1_R1_B,legend_label='sky', color='blue')

#for place in range(0,len(line_locations_region_1)):
    #q.line([range1_array[place,0]+0.03,range1_array[place,0]+0.03], [1.06,0.60],color='black',line_width=1,name=str(range1_array[place,1]), legend_label='Strong Telluric Features That Cannot Be Fully Processed')
#for place in range(0,len(two_line_locations_region_1)):
    #q.line([two_range1_array[place,0]+0.03,two_range1_array[place,0]+0.03], [1.06,0.60],color='black',line_width=1,name=str(two_range1_array[place,1]))

q.legend.location = "bottom_right"

show(q)

## Night One Region One C

In [75]:
vnorm_N1_R1_C = -.01+(((vflux_N1_R1_C-(np.min(vflux_N1_R1_C)))*(1+.01))/ vrange_N1_R1_C)
snorm_N1_R1_C = -.01+(((sflux_N1_R1_C-(np.min(sflux_N1_R1_C)))*(1+.01))/ srange_N1_R1_C)

In [76]:
print(min(R1_H2O))
print(R1_H2O.index(0.057861))
print(R1_lambda[3683])

#print(vlambda_N1_R1_C[1900:2000])
print(min(vnorm_N1_R1_C[1900:2000]))

H2Oexp_N1_R1_C = np.log(min(vnorm_N1_R1_C[1900:2000]))/np.log(min(R1_H2O))
#H2Oexp_N1_R1_C = 0.1
print(H2Oexp_N1_R1_C)

0.057861
3683
1121.229305
0.2444961162277628
0.4942801254172895


In [77]:
print(min(R1_O3))
print(R1_O3.index(0.0513534))
print(R1_lambda[3475])

#print(vlambda_N1_R1_C[1800:1900])
print(min(vnorm_N1_R1_C[1800:1900]))

#O3exp_N1_R1_C = np.log(min(vnorm_N1_R1_C[1800:1900]))/np.log(min(R1_O3))
O3exp_N1_R1_C = 0.55
print(O3exp_N1_R1_C)

0.0513534
3475
1120.762971
0.42238223816842574
0.55


In [78]:
N2Oexp_N1_R1_C = 0.1
print(N2Oexp_N1_R1_C)

0.1


In [79]:
CH4exp_N1_R1_C = 0.1
print(CH4exp_N1_R1_C)

0.1


In [80]:
k_N1_R1_C = 1.8 #Gaussian sigma of resolution broadening, in pixels
lambshift_N1_R1_C = 0.008 #empirical wavenumber shift to improve residuals"

In [81]:
#multiplying the spectra by their respective exponents
H2O_N1_R1_C=[]
O3_N1_R1_C=[]
N2O_N1_R1_C=[]
CH4_N1_R1_C=[]
for number in R1_H2O:
    H2O_N1_R1_C.append(number**H2Oexp_N1_R1_C)
for number in R1_O3:
    O3_N1_R1_C.append(number**O3exp_N1_R1_C)
for number in R1_N2O:
    N2O_N1_R1_C.append(number**N2Oexp_N1_R1_C)
for number in R1_CH4:
    CH4_N1_R1_C.append(number**CH4exp_N1_R1_C)

#multiplying the exponentiated spectra together to create a new composite spectrum
tot_N1_R1_C = np.array(H2O_N1_R1_C)*np.array(O3_N1_R1_C)*np.array(N2O_N1_R1_C)*np.array(CH4_N1_R1_C)

#gaussian smoothing the spectrum
gaus_N1_R1_C = gaussian_filter(tot_N1_R1_C,k_N1_R1_C)

#interpolating the spectrum over the wavenumber range
int_N1_R1_C = scipy.interpolate.interp1d(R1_lambdads, gaus_N1_R1_C)
intfunc_N1_R1_C = int_N1_R1_C(vlambda_N1_R1_C)
tell_N1_R1_C = shift(intfunc_N1_R1_C, lambshift_N1_R1_C)

#normalizing the model
normtell_N1_R1_C = tell_N1_R1_C / np.median(tell_N1_R1_C)

#dividing the spectrum by the model
vproc_N1_R1_C = np.divide(vnorm_N1_R1_C, normtell_N1_R1_C)

/var/folders/0q/22yph8ws4w999mfg8_bqcg840000gn/T/ipykernel_2872/3022568511.py:30: RuntimeWarning: divide by zero encountered in divide
  vproc_N1_R1_C = np.divide(vnorm_N1_R1_C, normtell_N1_R1_C)


In [82]:
#plotting the processed file
TOOLTIPS = [ ("x","$x{‘00.00000000’}"),("y","$y{‘00.000’}"), ("strength", "$name")]
q = figure(title="Night One WnRegion One GeoRegion C",sizing_mode="stretch_width",
    height=500,tooltips=TOOLTIPS,
    x_axis_label="Wavenumber (cm⁻¹)",
    y_axis_label="Radiance (erg·s⁻¹·cm⁻²·sr⁻¹)")

#q.line(vlambda_N1_R1_C, normtell_N1_R1_C -0.1, legend_label='corrected telluric', color='orange')
q.line(vlambda_N1_R1_C, vproc_N1_R1_C + 0.1, legend_label='venus processed', color='green')
q.line(vlambda_N1_R1_C, vnorm_N1_R1_C, legend_label='venus unprocessed', color='red')
#q.line(vlambda_N1_R1_C, snorm_N1_R1_C,legend_label='sky', color='blue')

q.x_range = Range1d(1121.4, 1121.8)

#for place in range(0,len(line_locations_region_1)):
    #q.line([range1_array[place,0]+0.03,range1_array[place,0]+0.03], [1.06,0.60],color='black',line_width=1,name=str(range1_array[place,1]), legend_label='Strong Telluric Features That Cannot Be Fully Processed')
#for place in range(0,len(two_line_locations_region_1)):
    #q.line([two_range1_array[place,0]+0.03,two_range1_array[place,0]+0.03], [1.06,0.60],color='black',line_width=1,name=str(two_range1_array[place,1]))

q.legend.location = "bottom_right"

show(q)

In [83]:
#plotting the processed file
TOOLTIPS = [ ("x","$x{‘00.00000000’}"),("y","$y{‘00.000’}"), ("strength", "$name")]
q = figure(title="Suspected Features",sizing_mode="stretch_width",
    height=500,tooltips=TOOLTIPS,    
    x_axis_label="Wavenumber (cm⁻¹)",
    y_axis_label="Radiance (erg·s⁻¹·cm⁻²·sr⁻¹)")

q.line(vlambda_N1_R1_A[560:590], vproc_N1_R1_A[560:590] - 0.1, legend_label='Venus Processed Night One A', color='red')
q.line(vlambda_N1_R1_B[560:590], vproc_N1_R1_B[560:590] - 0.175, legend_label='Venus Processed Night One B', color='green')
q.line(vlambda_N1_R1_C[560:590], vproc_N1_R1_C[560:590], legend_label='Venus Processed Night One C', color='blue')
q.line(vlambda_N1_R1_tot[560:590], vproc_N1_R1_tot[560:590], legend_label='Venus Processed Night One Total', color='black')
q.line(vlambda_N2_R1_tot[385:415], vproc_N2_R1_tot[385:415], legend_label='Venus Processed Night Two Total', color='orange')


#for place in range(0,len(line_locations_region_1)):
    #q.line([range1_array[place,0]+0.03,range1_array[place,0]+0.03], [1.06,0.60],color='black',line_width=1,name=str(range1_array[place,1]), legend_label='Strong Telluric Features That Cannot Be Fully Processed')
#for place in range(0,len(two_line_locations_region_1)):
    #q.line([two_range1_array[place,0]+0.03,two_range1_array[place,0]+0.03], [1.06,0.60],color='black',line_width=1,name=str(two_range1_array[place,1]))

q.legend.location = "bottom_right"

show(q)

# Polynomial Fit

## Night One Region One Total

In [84]:
threshold_1 = 0.93

# Get the indices using list comprehension
indices = [i for i, x in enumerate(vproc_N1_R1_tot) if x < threshold_1]

def indices_to_ranges(indices):
    if len(indices) == 0:
        return []
    
    ranges = []
    start = indices[0]
    end = indices[0]
    
    for i in range(1, len(indices)):
        if indices[i] == end + 1:  # Consecutive
            end = indices[i]
        else:  # Gap found, save current range and start new one
            ranges.append((start, end + 1))  # +1 because slicing is exclusive
            start = indices[i]
            end = indices[i]
    
    ranges.append((start, end + 1))
    return ranges

#print(f"The list is: {vlambda_N1_R1_tot}")
#print(f"The threshold is: {threshold}")
#print(f"Indices less than the threshold: {indices}")
#print(len(indices))
#print(len(vlambda_N1_R1_tot))

#print(vproc_N1_R1_tot)

In [85]:
np.set_printoptions(threshold=sys.maxsize)

mask = np.ones(len(vlambda_N1_R1_tot), dtype=bool)

keep = indices_to_ranges(indices)

for start, end in keep:
    mask[start:end] = False

masked_wavenumber = np.ma.masked_array(vlambda_N1_R1_tot, mask=mask)
masked_spectrum = np.ma.masked_array(vproc_N1_R1_tot, mask=mask)

#print(average_wavenumber_ds)
#print(masked_wavenumber)
#print(masked_spectrum)

### Just Right Fit

In [86]:
class KnotSplineModel(object):
    """
    This uses a B-spline of n degree, which is a a piecewise polynomial function of degree n that is defined by a set of control points and a set of knots. The knots parameter
    specifies the number of INTERIOR knots to use for the fit.

    * Inspired by PyModelFit UniformCDFKnotSplineModel:
        https://pythonhosted.org/PyModelFit/
    """

    def __init__(self, nknots=3, degree=2, CDF=True):
        self.nknots = nknots
        self.degree = degree
        self.CDF = CDF

    def fitData(self, x, y, weights=None):
        """
        Fit a uniform knot spline model. There will be a uniform seperation
        between the internal knots except if CDF is True.

        When CDF is True, the seperation between the internal knots is set
        uniformly on the CDF (i.e. knots at the locations that place them
        unifomly on the histogram of x-values)
        """
        # Extract only the unmasked data points
        if isinstance(x, np.ma.MaskedArray):
            mask = ~x.mask if x.mask.any() else np.ones_like(x, dtype=bool)
            if isinstance(y, np.ma.MaskedArray):
                # Combine masks if both arrays are masked
                mask = mask & ~y.mask if y.mask.any() else mask
            x_data = x[mask].data
            y_data = y[mask].data
            if weights is not None:
                weights = weights[mask]
        else:
            x_data = x
            y_data = y
            
        # Sort the unmasked data
        sorti = np.argsort(x_data)
        x_data = np.array(x_data)[sorti]
        y_data = np.array(y_data)[sorti]
        if weights is not None:
            weights = weights[sorti]
        
        # Rest of your function with x_data and y_data instead of x and y
        if self.CDF:
            cdf,xcdf = np.histogram(x_data, bins=max(10,max(2*self.nknots,int(len(x_data)/10))))
            mask = cdf!=0
            cdf,xcdf = cdf[mask],xcdf[np.hstack((True,mask))]
            cdf = np.hstack((0,(1.0*np.cumsum(cdf))/np.sum(cdf)))
            self.iknots = np.interp(np.linspace(0,1,self.nknots+2)[1:-1],cdf,xcdf)
        else:
            self.iknots = np.linspace(x_data[0],x_data[-1],self.nknots+2)[1:-1]
    
        # Make sure x_data is strictly increasing by checking for duplicates
        # Small epsilon to add to duplicates
        eps = (x_data[-1] - x_data[0]) * 1e-10
        for i in range(1, len(x_data)):
            if x_data[i] <= x_data[i-1]:
                x_data[i] = x_data[i-1] + eps
    
        self.spline = LSQUnivariateSpline(x_data, y_data, t=self.iknots, k=int(self.degree), w=weights)

    def __call__(self, x):
        return self.spline(x)

ranges = indices_to_ranges(indices)
segment_sizes = [end - start for start, end in ranges]
print(f"Segment sizes: {segment_sizes}")
print(f"Min: {min(segment_sizes)}, Max: {max(segment_sizes)}")

Segment sizes: [237, 71, 1, 7, 55, 37, 384, 27, 1, 9, 398, 89, 132, 352, 34, 83, 52, 27, 329]
Min: 1, Max: 398


In [87]:
model = KnotSplineModel(nknots=3, degree=3, CDF=True) #One knot every 10 angstrom
model.fitData(masked_wavenumber, masked_spectrum)
y_fit = model(masked_wavenumber)

wavelength_grid = np.array(vlambda_N1_R1_tot) 

# Filter out segments that are too small
min_segment_size = 25  # Minimum points needed for stable spline fitting
spectral_segments_ranges = [
    wavelength_grid[start:end] 
    for start, end in indices_to_ranges(indices)
    if end - start >= min_segment_size
]

print(f"Number of valid segments: {len(spectral_segments_ranges)}")
print(f"Segment sizes: {[len(seg) for seg in spectral_segments_ranges]}")

total_spline_fit = []
total_fxt_fitted = []
total_fxt_fitted_error = []
all_masked_wavelengths = []

# Initialize lists for this iteration
total_spline_fit_i = []
fxt_fitted_i = []
fxt_fitted_i_error = []

k = 0  # Assuming you want to use the first file

for i in spectral_segments_ranges:
    # Get the minimum and maximum values of each segment
    min_val = i[0]
    max_val = i[-1]
    
    # Mask wavelengths outside the desired range
    idx_within_range = (wavelength_grid >= min_val) & (wavelength_grid <= max_val)
    masked_wavelengths = wavelength_grid[idx_within_range]
    masked_values = vproc_N1_R1_tot[idx_within_range]
    #masked_values_error = div_expo_master_out_error[k][idx_within_range]
    
    # Adaptive knots based on segment size
    nknots = min(20, len(masked_wavelengths) // 10)  # At least 10 points per knot
    nknots = max(4, nknots)  # At least 4 knots
    print(nknots)
    
    model = KnotSplineModel(nknots=nknots, degree=3, CDF=True)
    model.fitData(masked_wavelengths, masked_values)
    y_fit = model(masked_wavelengths)
    total_spline_fit_i.extend(y_fit)
    fxt_fitted_i.extend(masked_values)
    #fxt_fitted_i_error.extend(masked_values_error)
    all_masked_wavelengths.extend(masked_wavelengths)  # Collect wavelengths
    
total_spline_fit.append(total_spline_fit_i)
total_fxt_fitted.append(fxt_fitted_i)
total_fxt_fitted_error.append(fxt_fitted_i_error)
    
print(f"Total points fitted: {len(total_spline_fit_i)}")
print(f"Number of segments: {len(total_spline_fit)}")

#maybe = np.divide(vproc_N1_R1_tot, total_spline_fit[0])

Number of valid segments: 15
Segment sizes: [237, 71, 55, 37, 384, 27, 398, 89, 132, 352, 34, 83, 52, 27, 329]
20
7
5
4
20
4
20
8
13
20
4
8
5
4
20
Total points fitted: 2307
Number of segments: 1


In [88]:
#plotting the processed file
TOOLTIPS = [ ("x","$x{‘00.00000000’}"),("y","$y{‘00.000’}"), ("strength", "$name")]
q = figure(title="Suspected Features",sizing_mode="stretch_width",
    height=500,tooltips=TOOLTIPS,    
    x_axis_label="Wavenumber (cm⁻¹)",
    y_axis_label="Radiance (erg·s⁻¹·cm⁻²·sr⁻¹)")

q.line(all_masked_wavelengths, total_spline_fit[0], legend_label='baseline fit', color='green')
q.line(vlambda_N1_R1_tot, vproc_N1_R1_tot + 0.1, legend_label='venus processed', color='red')

q.legend.location = "bottom_right"

show(q)

### Underfit

In [89]:
under_model = KnotSplineModel(nknots=20, degree=5, CDF=True) #One knot every 10 angstrom
under_model.fitData(masked_wavenumber, masked_spectrum)
under_y_fit = under_model(masked_wavenumber)

under_wavelength_grid = np.array(vlambda_N1_R1_tot) 

# Filter out segments that are too small
under_min_segment_size = 25  # Minimum points needed for stable spline fitting
under_spectral_segments_ranges = [
    under_wavelength_grid[start:end] 
    for start, end in indices_to_ranges(indices)
    if end - start >= under_min_segment_size
]

print(f"Number of valid segments: {len(under_spectral_segments_ranges)}")
print(f"Segment sizes: {[len(seg) for seg in under_spectral_segments_ranges]}")

under_total_spline_fit = []
under_total_fxt_fitted = []
under_total_fxt_fitted_error = []
under_all_masked_wavelengths = []

# Initialize lists for this iteration
under_total_spline_fit_i = []
under_fxt_fitted_i = []
under_fxt_fitted_i_error = []

k = 1  # Assuming you want to use the first file

for i in under_spectral_segments_ranges:
    # Get the minimum and maximum values of each segment
    min_val = i[0]
    max_val = i[-1]
    
    # Mask wavelengths outside the desired range
    under_idx_within_range = (under_wavelength_grid >= min_val) & (under_wavelength_grid <= max_val)
    under_masked_wavelengths = under_wavelength_grid[under_idx_within_range]
    under_masked_values = vproc_N1_R1_tot[under_idx_within_range]
    #masked_values_error = div_expo_master_out_error[k][idx_within_range]
    
    # Adaptive knots based on segment size
    under_nknots = min(20, len(under_masked_wavelengths) // 10)  # At least 10 points per knot
    under_nknots = max(4, under_nknots)  # At least 4 knots
    
    under_model = KnotSplineModel(nknots=25, degree=1, CDF=True)
    under_model.fitData(under_masked_wavelengths, under_masked_values)
    under_y_fit = under_model(under_masked_wavelengths)
    under_total_spline_fit_i.extend(under_y_fit)
    under_fxt_fitted_i.extend(under_masked_values)
    #fxt_fitted_i_error.extend(masked_values_error)
    under_all_masked_wavelengths.extend(under_masked_wavelengths)  # Collect wavelengths
    
under_total_spline_fit.append(under_total_spline_fit_i)
under_total_fxt_fitted.append(under_fxt_fitted_i)
under_total_fxt_fitted_error.append(under_fxt_fitted_i_error)
    
print(f"Total points fitted: {len(under_total_spline_fit_i)}")
print(f"Number of segments: {len(under_total_spline_fit)}")

#maybe = np.divide(vproc_N1_R1_tot, total_spline_fit[0])

Number of valid segments: 15
Segment sizes: [237, 71, 55, 37, 384, 27, 398, 89, 132, 352, 34, 83, 52, 27, 329]
Total points fitted: 2307
Number of segments: 1


### Overfit 

In [90]:
over_model = KnotSplineModel(nknots=20, degree=5, CDF=True) #One knot every 10 angstrom
over_model.fitData(masked_wavenumber, masked_spectrum)
over_y_fit = over_model(masked_wavenumber)

over_wavelength_grid = np.array(vlambda_N1_R1_tot) 

# Filter out segments that are too small
over_min_segment_size = 25  # Minimum points needed for stable spline fitting
over_spectral_segments_ranges = [
    over_wavelength_grid[start:end] 
    for start, end in indices_to_ranges(indices)
    if end - start >= over_min_segment_size
]

print(f"Number of valid segments: {len(over_spectral_segments_ranges)}")
print(f"Segment sizes: {[len(seg) for seg in over_spectral_segments_ranges]}")

over_total_spline_fit = []
over_total_fxt_fitted = []
over_total_fxt_fitted_error = []
over_all_masked_wavelengths = []

# Initialize lists for this iteration
over_total_spline_fit_i = []
over_fxt_fitted_i = []
over_fxt_fitted_i_error = []

k = 1  # Assuming you want to use the first file

for i in over_spectral_segments_ranges:
    # Get the minimum and maximum values of each segment
    min_val = i[0]
    max_val = i[-1]
    
    # Mask wavelengths outside the desired range
    over_idx_within_range = (over_wavelength_grid >= min_val) & (over_wavelength_grid <= max_val)
    over_masked_wavelengths = over_wavelength_grid[over_idx_within_range]
    over_masked_values = vproc_N1_R1_tot[over_idx_within_range]
    #masked_values_error = div_expo_master_out_error[k][idx_within_range]
    
    # Adaptive knots based on segment size
    over_nknots = min(20, len(over_masked_wavelengths) // 10)  # At least 10 points per knot
    over_nknots = max(4, over_nknots)  # At least 4 knots
    
    over_model = KnotSplineModel(nknots=4, degree=3, CDF=True)
    over_model.fitData(over_masked_wavelengths, over_masked_values)
    over_y_fit = over_model(over_masked_wavelengths)
    over_total_spline_fit_i.extend(over_y_fit)
    over_fxt_fitted_i.extend(over_masked_values)
    #fxt_fitted_i_error.extend(masked_values_error)
    over_all_masked_wavelengths.extend(over_masked_wavelengths)  # Collect wavelengths
    
over_total_spline_fit.append(over_total_spline_fit_i)
over_total_fxt_fitted.append(over_fxt_fitted_i)
over_total_fxt_fitted_error.append(over_fxt_fitted_i_error)
    
print(f"Total points fitted: {len(over_total_spline_fit_i)}")
print(f"Number of segments: {len(over_total_spline_fit)}")

Number of valid segments: 15
Segment sizes: [237, 71, 55, 37, 384, 27, 398, 89, 132, 352, 34, 83, 52, 27, 329]
Total points fitted: 2307
Number of segments: 1


In [91]:
#plotting the processed file
TOOLTIPS = [ ("x","$x{‘00.00000000’}"),("y","$y{‘00.000’}"), ("strength", "$name")]
q = figure(title="2025A Baseline Fits",sizing_mode="stretch_width",
    height=500,tooltips=TOOLTIPS,    
    x_axis_label="Wavenumber (cm⁻¹)",
    y_axis_label="Intensity")

q.line(over_all_masked_wavelengths, np.array(over_total_spline_fit[0])+0.1, legend_label='nknots = 4, degree = 3', color='red')
q.line(under_all_masked_wavelengths, np.array(under_total_spline_fit[0])-0.1, legend_label='nknots = 25, degree = 1', color='blue')
q.line(all_masked_wavelengths, total_spline_fit[0], legend_label='nknots = 20, degree = 3', color='green')

q.y_range = Range1d(0.65, 1.05)

q.legend.location = "bottom_right"

show(q)

### Final File

In [92]:
#interpolating the spectrum over the wavenumber range
fit_intfunc_N1_R1_tot = scipy.interpolate.interp1d(over_all_masked_wavelengths, total_spline_fit[0])
fit_int_N1_R1_tot = fit_intfunc_N1_R1_tot(vlambda_N1_R1_tot)
fit_int_N1_R1_tot[fit_int_N1_R1_tot < 0] = 0.0000001
vproc_N1_R1_tot[vproc_N1_R1_tot < 0] = 0.00001

fit_N1_R1_tot = np.divide(vproc_N1_R1_tot,fit_int_N1_R1_tot)

#interpolating the spectrum over the wavenumber range
over_fit_intfunc_N1_R1_tot = scipy.interpolate.interp1d(over_all_masked_wavelengths, over_total_spline_fit[0])
over_fit_int_N1_R1_tot = over_fit_intfunc_N1_R1_tot(vlambda_N1_R1_tot)
over_fit_int_N1_R1_tot[over_fit_int_N1_R1_tot < 0] = 0.0000001
vproc_N1_R1_tot[vproc_N1_R1_tot < 0] = 0.00001

over_fit_N1_R1_tot = np.divide(vproc_N1_R1_tot,over_fit_int_N1_R1_tot)

#interpolating the spectrum over the wavenumber range
under_fit_intfunc_N1_R1_tot = scipy.interpolate.interp1d(under_all_masked_wavelengths, under_total_spline_fit[0])
under_fit_int_N1_R1_tot = under_fit_intfunc_N1_R1_tot(vlambda_N1_R1_tot)
under_fit_int_N1_R1_tot[under_fit_int_N1_R1_tot < 0] = 0.0000001
vproc_N1_R1_tot[vproc_N1_R1_tot < 0] = 0.00001

under_fit_N1_R1_tot = np.divide(vproc_N1_R1_tot,under_fit_int_N1_R1_tot)

In [116]:
#plotting the processed file
TOOLTIPS = [ ("x","$x{‘00.00000000’}"),("y","$y{‘00.000’}"), ("strength", "$name")]
q = figure(title="2025A - Polynomial Fit",sizing_mode="stretch_width",
    height=500,tooltips=TOOLTIPS,    
    x_axis_label="Wavenumber (cm⁻¹)",
    y_axis_label="Intensity",
    y_range=(0.77, 0.92))

q.line(vlambda_N1_R1_tot, fit_N1_R1_tot - 0.1, legend_label='2025A - Fully Processed', color='blue')
#q.line(vlambda_N1_R1_tot, under_fit_N1_R1_tot + 0.1, legend_label='venus fit - under', color='black')
#q.line(vlambda_N1_R1_tot, over_fit_N1_R1_tot, legend_label='venus fit - over', color='gray')
q.line(vlambda_N1_R1_tot, fit_int_N1_R1_tot-0.05, legend_label='2025A - Baseline Fit', color='green')
q.line(vlambda_N1_R1_tot, vproc_N1_R1_tot-0.1, legend_label='2025A - Sky Divided', color='red')

#for place in range(0,len(line_locations_region_1)):
    #q.line([range1_array[place,0]+0.03,range1_array[place,0]+0.03], [1.06,0.60],color='yellow',line_width=1,name=str(range1_array[place,1]), legend_label='H2O features that cannot be fully processed')
#for place in range(0,len(two_line_locations_region_1)):
    #q.line([two_range1_array[place,0]+0.03,two_range1_array[place,0]+0.03], [1.06,0.60],color='orange',line_width=1,name=str(two_range1_array[place,1]), legend_label='O3 features that cannot be fully processed')


q.legend.location = "bottom_right"

show(q)

### Unproccessed Data Fit

In [94]:
normunproc_N1_R1_tot  = vflux_N1_R1_tot / np.median(vflux_N1_R1_tot)
unproc_fit_N1_R1_tot = np.divide(normunproc_N1_R1_tot,fit_int_N1_R1_tot)

In [95]:
#plotting the processed file
TOOLTIPS = [ ("x","$x{‘00.00000000’}"),("y","$y{‘00.000’}"), ("strength", "$name")]
q = figure(title="Polynomial Fit",sizing_mode="stretch_width",
    height=500,tooltips=TOOLTIPS,    
    x_axis_label="Wavenumber (cm⁻¹)",
    y_axis_label="Radiance (erg·s⁻¹·cm⁻²·sr⁻¹)",
    y_range=(0, 1.2))

q.line(vlambda_N1_R1_tot, unproc_fit_N1_R1_tot, legend_label='unprocessed venus fit - just right', color='blue')
q.line(vlambda_N1_R1_tot, normtell_N1_R1_tot, legend_label='corrected telluric', color='orange')

#for place in range(0,len(line_locations_region_1)):
    #q.line([range1_array[place,0]+0.03,range1_array[place,0]+0.03], [1.06,0.60],color='yellow',line_width=1,name=str(range1_array[place,1]), legend_label='H2O features that cannot be fully processed')
#for place in range(0,len(two_line_locations_region_1)):
    #q.line([two_range1_array[place,0]+0.03,two_range1_array[place,0]+0.03], [1.06,0.60],color='orange',line_width=1,name=str(two_range1_array[place,1]), legend_label='O3 features that cannot be fully processed')

q.line(1114.808, [1.2,1.0],color='black',line_width=1, legend_label='marks')
q.line(1115.384, [1.2,1.0],color='black',line_width=1)
q.line(1115.474, [1.2,1.0],color='black',line_width=1)
q.line(1116.04489, [1.2,1.0],color='black',line_width=1)
q.line(1116.135, [1.2,1.0],color='black',line_width=1)
q.line(1116.5928, [1.2,1.0],color='black',line_width=1)
q.line(1116.805, [1.2,1.0],color='black',line_width=1)
q.line(1117.471, [1.2,1.0],color='black',line_width=1)
q.line(1120.1808, [1.2,1.0],color='black',line_width=1)

q.legend.location = "bottom_right"

show(q)

## Night One Region Two

In [96]:
vnorm_N1_R2_tot = -.01+(((vflux_N1_R2_tot-(np.min(vflux_N1_R2_tot)))*(1+.01))/ vrange_N1_R2_tot)
snorm_N1_R2_tot = -.01+(((sflux_N1_R2_tot-(np.min(sflux_N1_R2_tot)))*(1+.01))/ srange_N1_R2_tot)

In [97]:
threshold_1 = 0.93

# Get the indices using list comprehension
indices = [i for i, x in enumerate(vnorm_N1_R2_tot) if x < threshold_1]

def indices_to_ranges(indices):
    if len(indices) == 0:
        return []
    
    ranges = []
    start = indices[0]
    end = indices[0]
    
    for i in range(1, len(indices)):
        if indices[i] == end + 1:  # Consecutive
            end = indices[i]
        else:  # Gap found, save current range and start new one
            ranges.append((start, end + 1))  # +1 because slicing is exclusive
            start = indices[i]
            end = indices[i]
    
    ranges.append((start, end + 1))
    return ranges

#print(f"The list is: {vlambda_N1_R1_tot}")
#print(f"The threshold is: {threshold}")
#print(f"Indices less than the threshold: {indices}")
#print(len(indices))
#print(len(vlambda_N1_R1_tot))

#print(vproc_N1_R1_tot)

In [98]:
np.set_printoptions(threshold=sys.maxsize)

mask = np.ones(len(vlambda_N1_R2_tot), dtype=bool)

keep = indices_to_ranges(indices)

for start, end in keep:
    mask[start:end] = False

masked_wavenumber_R2 = np.ma.masked_array(vlambda_N1_R2_tot, mask=mask)
masked_spectrum_R2 = np.ma.masked_array(vnorm_N1_R2_tot, mask=mask)

#print(average_wavenumber_ds)
#print(masked_wavenumber)
#print(masked_spectrum)

In [99]:
class KnotSplineModel(object):
    """
    This uses a B-spline of n degree, which is a a piecewise polynomial function of degree n that is defined by a set of control points and a set of knots. The knots parameter
    specifies the number of INTERIOR knots to use for the fit.

    * Inspired by PyModelFit UniformCDFKnotSplineModel:
        https://pythonhosted.org/PyModelFit/
    """

    def __init__(self, nknots=3, degree=2, CDF=True):
        self.nknots = nknots
        self.degree = degree
        self.CDF = CDF

    def fitData(self, x, y, weights=None):
        """
        Fit a uniform knot spline model. There will be a uniform seperation
        between the internal knots except if CDF is True.

        When CDF is True, the seperation between the internal knots is set
        uniformly on the CDF (i.e. knots at the locations that place them
        unifomly on the histogram of x-values)
        """
        # Extract only the unmasked data points
        if isinstance(x, np.ma.MaskedArray):
            mask = ~x.mask if x.mask.any() else np.ones_like(x, dtype=bool)
            if isinstance(y, np.ma.MaskedArray):
                # Combine masks if both arrays are masked
                mask = mask & ~y.mask if y.mask.any() else mask
            x_data = x[mask].data
            y_data = y[mask].data
            if weights is not None:
                weights = weights[mask]
        else:
            x_data = x
            y_data = y
            
        # Sort the unmasked data
        sorti = np.argsort(x_data)
        x_data = np.array(x_data)[sorti]
        y_data = np.array(y_data)[sorti]
        if weights is not None:
            weights = weights[sorti]
        
        # Rest of your function with x_data and y_data instead of x and y
        if self.CDF:
            cdf,xcdf = np.histogram(x_data, bins=max(10,max(2*self.nknots,int(len(x_data)/10))))
            mask = cdf!=0
            cdf,xcdf = cdf[mask],xcdf[np.hstack((True,mask))]
            cdf = np.hstack((0,(1.0*np.cumsum(cdf))/np.sum(cdf)))
            self.iknots = np.interp(np.linspace(0,1,self.nknots+2)[1:-1],cdf,xcdf)
        else:
            self.iknots = np.linspace(x_data[0],x_data[-1],self.nknots+2)[1:-1]
    
        # Make sure x_data is strictly increasing by checking for duplicates
        # Small epsilon to add to duplicates
        eps = (x_data[-1] - x_data[0]) * 1e-10
        for i in range(1, len(x_data)):
            if x_data[i] <= x_data[i-1]:
                x_data[i] = x_data[i-1] + eps
    
        self.spline = LSQUnivariateSpline(x_data, y_data, t=self.iknots, k=int(self.degree), w=weights)

    def __call__(self, x):
        return self.spline(x)

ranges = indices_to_ranges(indices)
segment_sizes = [end - start for start, end in ranges]
print(f"Segment sizes: {segment_sizes}")
print(f"Min: {min(segment_sizes)}, Max: {max(segment_sizes)}")

Segment sizes: [1566]
Min: 1566, Max: 1566


In [100]:
model = KnotSplineModel(nknots=3, degree=3, CDF=True) #One knot every 10 angstrom
model.fitData(masked_wavenumber_R2, masked_spectrum_R2)
y_fit = model(masked_wavenumber_R2)

wavelength_grid = np.array(vlambda_N1_R2_tot) 

# Filter out segments that are too small
min_segment_size = 25  # Minimum points needed for stable spline fitting
spectral_segments_ranges = [
    wavelength_grid[start:end] 
    for start, end in indices_to_ranges(indices)
    if end - start >= min_segment_size
]

print(f"Number of valid segments: {len(spectral_segments_ranges)}")
print(f"Segment sizes: {[len(seg) for seg in spectral_segments_ranges]}")

total_spline_fit = []
total_fxt_fitted = []
total_fxt_fitted_error = []
all_masked_wavelengths = []

# Initialize lists for this iteration
total_spline_fit_i = []
fxt_fitted_i = []
fxt_fitted_i_error = []

k = 0  # Assuming you want to use the first file

for i in spectral_segments_ranges:
    # Get the minimum and maximum values of each segment
    min_val = i[0]
    max_val = i[-1]
    
    # Mask wavelengths outside the desired range
    idx_within_range = (wavelength_grid >= min_val) & (wavelength_grid <= max_val)
    masked_wavelengths = wavelength_grid[idx_within_range]
    masked_values = vnorm_N1_R2_tot[idx_within_range]
    #masked_values_error = div_expo_master_out_error[k][idx_within_range]
    
    # Adaptive knots based on segment size
    nknots = min(20, len(masked_wavelengths) // 10)  # At least 10 points per knot
    nknots = max(4, nknots)  # At least 4 knots
    
    model = KnotSplineModel(nknots=55, degree=3, CDF=True)
    model.fitData(masked_wavelengths, masked_values)
    y_fit = model(masked_wavelengths)
    total_spline_fit_i.extend(y_fit)
    fxt_fitted_i.extend(masked_values)
    #fxt_fitted_i_error.extend(masked_values_error)
    all_masked_wavelengths.extend(masked_wavelengths)  # Collect wavelengths
    
total_spline_fit.append(total_spline_fit_i)
total_fxt_fitted.append(fxt_fitted_i)
total_fxt_fitted_error.append(fxt_fitted_i_error)
    
print(f"Total points fitted: {len(total_spline_fit_i)}")
print(f"Number of segments: {len(total_spline_fit)}")

Number of valid segments: 1
Segment sizes: [1566]
Total points fitted: 1566
Number of segments: 1


In [101]:
fit_intfunc_N1_R2_tot = scipy.interpolate.interp1d(all_masked_wavelengths, total_spline_fit[0])
fit_int_N1_R2_tot = fit_intfunc_N1_R2_tot(vlambda_N1_R2_tot)
fit_int_N1_R2_tot[fit_int_N1_R2_tot < 0] = 0.0000001
vnorm_N1_R2_tot[vnorm_N1_R2_tot < 0] = 0.00001

fit_N1_R2_tot = np.divide(vnorm_N1_R2_tot,fit_int_N1_R2_tot)

In [102]:
#plotting the processed file
TOOLTIPS = [ ("x","$x{‘00.00000000’}"),("y","$y{‘00.000’}"), ("strength", "$name")]
q = figure(title="Baseline Fits",sizing_mode="stretch_width",
    height=500,tooltips=TOOLTIPS,    
    x_axis_label="Wavenumber (cm⁻¹)",
    y_axis_label="Radiance (erg·s⁻¹·cm⁻²·sr⁻¹)")

q.line(all_masked_wavelengths, total_spline_fit[0], legend_label='baseline right fit', color='green')
q.line(vlambda_N1_R2_tot, vnorm_N1_R2_tot, legend_label='venus', color='red')
q.line(vlambda_N1_R2_tot, fit_N1_R2_tot, legend_label='fit', color='blue')

q.y_range = Range1d(0, 1.02)

q.legend.location = "bottom_right"

show(q)

# PSG Retrival 

In [103]:
N1_R1_tot = pd.DataFrame({'Wavenumber' : vlambda_N1_R1_tot,
                                'Flux' : fit_N1_R1_tot})
print(N1_R1_tot)
N1_R1_tot.to_csv('N1_R1_tot.tsv', index=False)

       Wavenumber   Flux
0     1113.956636  100.0
1     1113.960352  100.0
2     1113.964068  100.0
3     1113.967784  100.0
4     1113.971499  100.0
...           ...    ...
2380  1122.800216  100.0
2381  1122.803932  100.0
2382  1122.807648  100.0
2383  1122.811363  100.0
2384  1122.815079  100.0

[2385 rows x 2 columns]


In [104]:
N1_R2_tot = pd.DataFrame({'Wavenumber' : vlambda_N1_R2_tot,
                                'Flux' : vnorm_N1_R2_tot})
print(N1_R2_tot)
N1_R2_tot.to_csv('N1_R2_tot.tsv', index=False)

       Wavenumber     Flux
0     1204.004639  0.00001
1     1204.008655  0.00001
2     1204.012671  0.00001
3     1204.016687  0.00001
4     1204.020704  0.00001
...           ...      ...
1561  1210.273865  0.00001
1562  1210.277881  0.00001
1563  1210.281897  0.00001
1564  1210.285913  0.00001
1565  1210.289930  0.00001

[1566 rows x 2 columns]


In [105]:
#plotting the processed file
TOOLTIPS = [ ("x","$x{‘00.00000000’}"),("y","$y{‘00.000’}"), ("strength", "$name")]
q = figure(title="Night One Region Two",sizing_mode="stretch_width",
    height=500,tooltips=TOOLTIPS,    
    x_axis_label="Wavenumber (cm⁻¹)",
    y_axis_label="Intensity")

q.line(vlambda_N1_R2_tot, vnorm_N1_R2_tot, legend_label='venus', color='red')
q.line(vlambda_N1_R2_tot, snorm_N1_R2_tot, legend_label='venus', color='blue')

q.legend.location = "bottom_right"

show(q)

In [106]:
#plotting the processed file
TOOLTIPS = [ ("x","$x{‘00.00000000’}"),("y","$y{‘00.000’}"), ("strength", "$name")]
q = figure(title="PSG Model",sizing_mode="stretch_width",
    height=500,tooltips=TOOLTIPS,    
    x_axis_label="Wavenumber (cm⁻¹)",
    y_axis_label="Radiance (erg·s⁻¹·cm⁻²·sr⁻¹)")

q.line(N1_R1_tot_psg_1115_wn, N1_R1_tot_psg_1115_data, legend_label='data', color='blue')
#q.line(N1_R1_tot_psg_1115_wn, N1_R1_tot_psg_1115_model, legend_label='psg model', color='green')
q.line(N1_R1_tot_psg_11155_wn, N1_R1_tot_psg_11155_data, legend_label='data', color='blue')
#q.line(N1_R1_tot_psg_11155_wn, N1_R1_tot_psg_11155_model, legend_label='psg model', color='green')
q.line(N1_R1_tot_psg_1116_wn, N1_R1_tot_psg_1116_data, legend_label='data', color='blue')
#q.line(N1_R1_tot_psg_1116_wn, N1_R1_tot_psg_1116_model, legend_label='psg model', color='green')
q.line(N1_R1_tot_psg_11165_wn, N1_R1_tot_psg_11165_data, legend_label='data', color='blue')
#q.line(N1_R1_tot_psg_11165_wn, N1_R1_tot_psg_11165_model, legend_label='psg model', color='green')
q.line(N1_R1_tot_psg_1117_wn, N1_R1_tot_psg_1117_data, legend_label='data', color='blue')
#q.line(N1_R1_tot_psg_1117_wn, N1_R1_tot_psg_1117_model, legend_label='psg model', color='green')
q.line(N1_R1_tot_psg_11175_wn, N1_R1_tot_psg_11175_data, legend_label='data', color='blue')
#q.line(N1_R1_tot_psg_11175_wn, N1_R1_tot_psg_11175_model, legend_label='psg model', color='green')
q.line(N1_R1_tot_psg_1118_wn, N1_R1_tot_psg_1118_data, legend_label='data', color='blue')
#q.line(N1_R1_tot_psg_1118_wn, N1_R1_tot_psg_1118_model, legend_label='psg model', color='green')
q.line(N1_R1_tot_psg_11185_wn, N1_R1_tot_psg_11185_data, legend_label='data', color='blue')
#q.line(N1_R1_tot_psg_11185_wn, N1_R1_tot_psg_11185_model, legend_label='psg model', color='green')
q.line(N1_R1_tot_psg_1119_wn, N1_R1_tot_psg_1119_data, legend_label='data', color='blue')
#q.line(N1_R1_tot_psg_1119_wn, N1_R1_tot_psg_1119_model, legend_label='psg model', color='green')
q.line(N1_R1_tot_psg_11195_wn, N1_R1_tot_psg_11195_data, legend_label='data', color='blue')
#q.line(N1_R1_tot_psg_11195_wn, N1_R1_tot_psg_11195_model, legend_label='psg model', color='green')
q.line(N1_R1_tot_psg_1120_wn, N1_R1_tot_psg_1120_data, legend_label='data', color='blue')
#q.line(N1_R1_tot_psg_1120_wn, N1_R1_tot_psg_1120_model, legend_label='psg model', color='green')
q.line(N1_R1_tot_psg_11205_wn, N1_R1_tot_psg_11205_data, legend_label='data', color='blue')
#q.line(N1_R1_tot_psg_11205_wn, N1_R1_tot_psg_11205_model, legend_label='psg model', color='green')
q.line(N1_R1_tot_psg_1121_wn, N1_R1_tot_psg_1121_data, legend_label='data', color='blue')
#q.line(N1_R1_tot_psg_1121_wn, N1_R1_tot_psg_1121_model, legend_label='psg model', color='green')
q.line(N1_R1_tot_psg_11215_wn, N1_R1_tot_psg_11215_data, legend_label='data', color='blue')
#q.line(N1_R1_tot_psg_11215_wn, N1_R1_tot_psg_11215_model, legend_label='psg model', color='green')
q.line(N1_R1_tot_psg_1122_wn, N1_R1_tot_psg_1122_data, legend_label='data', color='blue')
#q.line(N1_R1_tot_psg_1122_wn, N1_R1_tot_psg_1122_model, legend_label='psg model', color='green')
q.line(N1_R1_tot_psg_11225_wn, N1_R1_tot_psg_11225_data, legend_label='data', color='blue')
#q.line(N1_R1_tot_psg_11225_wn, N1_R1_tot_psg_11225_model, legend_label='psg model', color='green')

q.line(vlambda_N1_R1_tot, fit_N1_R1_tot, legend_label='fit', color='orange')
q.line(vlambda_N1_R1_tot, normtell_N1_R1_tot, legend_label='corrected telluric', color='black')

#q.x_range = Range1d(1121, 1121.5)
q.y_range = Range1d(0.95, 1.02)

q.legend.location = "bottom_right"

show(q)

NameError: name 'N1_R1_tot_psg_1115_wn' is not defined

In [126]:
N1_2021 = []
N1_2021_lamb = [] 

with open('/Users/physicsstudent2/desktop/venus_research/Venus_Data_Confidential/2021 PSG/data_2021.csv', encoding='latin-1') as data_2021_file:  
    plots = csv.reader((line.replace('\x00', '') for line in data_2021_file), delimiter=',')

    for row in plots:
        if row and row[0] != 'wavelength':
            N1_2021_lamb.append(float(row[0]))
            N1_2021.append(float(row[1]))

In [168]:
import pandas as pd
from bokeh.plotting import figure, show
from bokeh.models import Legend

df = pd.read_csv('/Users/physicsstudent2/desktop/psg_rad_9.csv')

TOOLTIPS = [("Wavenumber", "$x{0.000000}"), ("Radiance", "$y{0.000000e+0}")]

p = figure(
    title="PSG Radiance Spectrum",
    sizing_mode="stretch_width",
    height=500,
    tooltips=TOOLTIPS,
    x_axis_label="Wavenumber (cm⁻¹)",
    y_axis_label="Radiance (W/sr/m²)"
)

p.line(df['Wave/freq'], df['Total'], legend_label='Total', color='black',  line_width=1.5)
#p.line(df['Wave/freq'], df['Noise'], legend_label='Noise', color='red',    line_width=1)
#p.line(df['Wave/freq'], df['Venus'], legend_label='Venus', color='orange', line_width=1)

p.legend.location = "bottom_right"
p.legend.click_policy = "hide"

show(p)

normal_total = df['Total'] / df['Total'].max()


In [169]:
#plotting the processed file
TOOLTIPS = [ ("x","$x{‘00.00000000’}"),("y","$y{‘00.000’}"), ("strength", "$name")]
q = figure(title="2025A vs. 2021A - Data",sizing_mode="stretch_width",
    height=500,tooltips=TOOLTIPS,    
    x_axis_label="Wavenumber (cm⁻¹)",
    y_axis_label="Intensity")

q.line(vlambda_N1_R1_tot, fit_N1_R1_tot + 0.01, legend_label='2025A - Fully Processed', color='blue')
q.line(N1_2021_lamb, N1_2021, legend_label='2021A - Fully Processed', color='red')
q.line(df['Wave/freq'], normal_total + 0.02, legend_label='PH3 - 10 ppb', color='black',  line_width=1.5)

#q.x_range = Range1d(1121, 1121.5)
q.y_range = Range1d(0.96, 1.02)

q.legend.location = "bottom_right"

show(q)